# Codex Work Product: YouTube Attention Manuscript Analysis

**Authoring label:** this notebook is the Codex-generated work product requested on 2026-05-29 for the YouTube descriptive manuscript. It is intentionally labeled so it can be compared against another model's notebook.

**Objective:** produce the core aggregate results, main figures, extended-data figures, and robustness checks needed for a platform-scale computational social science manuscript on public YouTube attention.

**Data-governance contract:** all row-level analysis is performed inside Databricks. The notebook writes aggregate Delta tables and publication figures only by default. Internal row-level scratch Delta tables are disabled unless `persist_row_level_scratch=true` is set for a controlled debugging run. It does not export bulk channel-, video-, comment-, or transcript-level data from the Databricks environment. Treemap source data intentionally includes only the labeled public channels and pooled cells needed to reproduce the manuscript figure; it is not a bulk channel export.

**Default behavior:** the notebook starts in `manifest_only` mode so reviewers and collaborators can inspect configuration without triggering expensive scans. Set `execution_mode` to `smoke`, `core`, or `full`; set `confirm_expensive=true` before running `core` or `full`.


## Analysis Map

This notebook is organized around the manuscript's display-item plan.

1. **Discovery saturation:** batch yields, threshold crossing, and benchmark overlap diagnostics.
2. **Whole-platform map:** language x category x channel composition and traffic-block heterogeneity.
3. **Subscriber proxy failure:** current attention versus subscribers, prediction envelopes, and variance by language/category.
4. **Threshold capture:** channel/view shares above 1k, 10k, 100k, and 1M subscribers; rank-capture and Lorenz diagnostics.
5. **Age/incumbency structure:** channel-start distributions by attention block, with source/proxy flags.
6. **Production and format:** uploads, long-form versus Shorts, and production/attention divergence.
7. **Robustness and reviewer checks:** alternative attention windows, suspicious-channel exclusions, label sources, category sources, bounds, and sampling uncertainty.

The code is modular so a Databricks reviewer can run a smoke test first, inspect aggregate outputs, and then approve the full run.


## Project Decisions Incorporated

This draft reflects the May 2026 project guidance that the canonical manuscript attention metric will be built from **weekly snapshots of lifetime channel views**. The Sunday snapshot job currently covers the Top of the Ocean sample, approximately all channels with at least 10,000 subscribers. A lightweight SQL probe on 2026-05-29 found that `prod_tads.youtube_too.yt_sl_channels_metrics` currently maxes at `capture_date=2026-05-01`, while `dev_sean.default.yt_channel_stats_full` has snapshot partitions on `2026-05-27` and `2026-05-28`. The later partition was much smaller in the probe and is treated as possibly partial. The notebook therefore prefers `yt_channel_stats_full`, but leaves `target_capture_date` blank by default and auto-selects the latest partition whose row count is at least 90% of the largest visible partition. Because the currently visible panel has adjacent-day partitions rather than a Sunday-to-Sunday pair, short-window fallbacks are disabled by default; the manuscript should use a real 7-day pair once the weekly panel has matured.

The `dev_sean.diagnostics.too_*` and `dev_sean.validation.*` layers are consumed as important evidence and QA surfaces, but not assumed authoritative. The notebook rebuilds core aggregates from snapshot/silver tables where possible, then compares those results to the TOO layer.

The category axis now prioritizes `dev_sean.default.backfill_channels.topic_categories`. A 0.1% sampled probe suggested that overall coverage is low because most rows are still `pending`, but rows with `status='done'` had substantial topic coverage. The notebook writes category coverage diagnostics and falls back to configured video-label columns only when topic categories are unavailable.


## Visualization Standards Used Here

The figure code uses conservative publication defaults: vector outputs, colorblind-safe palettes, direct labels where feasible, log scaling for heavy-tailed quantities, explicit denominators, and minimal chart furniture. The notebook treats figures as scientific evidence, not decoration.

References for style and file-preparation choices:

- Nature Portfolio figure preparation guidance: https://www.nature.com/nature/for-authors/final-submission
- Springer Nature manuscript and figure guidance: https://support.springernature.com/en/support/solutions/articles/6000210572-figure-formatting
- Science/AAAS figure preparation guidance: https://www.science.org/content/page/instructions-preparing-initial-manuscript
- Wong, B. 2011. *Color blindness*. Nature Methods.
- Cleveland, W. S. and McGill, R. 1984. *Graphical perception: theory, experimentation, and application to the development of graphical methods*.
- Rougier, Droettboom, and Bourne. 2014. *Ten simple rules for better figures*.

Practical conventions implemented below:

- Single-column width approx 90 mm; double-column width approx 180 mm.
- Save PNG for quick review plus PDF/SVG for manuscript assembly.
- Use bounded pandas extracts from aggregate tables only.
- Prefer rank/quantile summaries over unreadable raw scatterplots for platform-scale data.


## 0. Imports, Widgets, and Safety Guards


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import re
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    import matplotlib as mpl
    import matplotlib.pyplot as plt
except Exception as exc:  # pragma: no cover - Databricks runtime should have matplotlib or install it below.
    raise RuntimeError("matplotlib is required for manuscript figures. Set install_optional_packages=true if needed.") from exc

try:
    from pyspark.sql import DataFrame, Window
    from pyspark.sql import functions as F
    from pyspark.sql.types import BooleanType, DateType, DoubleType, IntegerType, LongType, StringType, StructField, StructType, TimestampType
    try:
        from pyspark.errors import AnalysisException
    except Exception:  # DBR compatibility
        from pyspark.sql.utils import AnalysisException
except Exception as exc:
    raise RuntimeError("This notebook is intended to run in Databricks or Databricks Connect with PySpark available.") from exc


def _in_databricks() -> bool:
    return "dbutils" in globals()


def _create_text_widget(name: str, default: str, label: Optional[str] = None) -> None:
    if _in_databricks():
        try:
            dbutils.widgets.text(name, default, label or name)
        except Exception:
            pass


def _get_widget(name: str, default: str) -> str:
    if _in_databricks():
        try:
            value = dbutils.widgets.get(name)
            return value if value not in {None, ""} else default
        except Exception:
            pass
    return os.environ.get(name.upper(), default)


def _get_bool_widget(name: str, default: bool) -> bool:
    raw = _get_widget(name, str(default)).strip().lower()
    return raw in {"1", "true", "t", "yes", "y", "on"}


def _get_int_widget(name: str, default: int) -> int:
    raw = _get_widget(name, str(default)).strip()
    return int(raw) if raw else default


def _get_float_widget(name: str, default: float) -> float:
    raw = _get_widget(name, str(default)).strip()
    return float(raw) if raw else default


def _safe_token(raw: str, default: str = "run") -> str:
    token = re.sub(r"[^A-Za-z0-9_]", "_", (raw or "").strip())
    token = re.sub(r"_+", "_", token).strip("_")
    return token or default


WIDGET_DEFAULTS = {
    # Run control.
    "execution_mode": "manifest_only",  # manifest_only, smoke, core, full
    "confirm_expensive": "false",
    "install_optional_packages": "false",
    "analysis_run_id": "",
    "random_seed": "20260529",
    "spark_shuffle_partitions": "800",
    "smoke_channel_limit": "50000",
    "write_aggregate_csv": "true",
    "persist_row_level_scratch": "false",
    "min_export_cell_count": "5",
    "fail_on_oversized_export": "true",
    "enable_displays": "true",
    # Source tables.
    "source_catalog": "prod_tads",
    "source_schema": "youtube_too",
    "channels_table": "yt_sl_channels",
    "channel_metrics_table": "yt_sl_channels_metrics",
    "channel_snapshot_table_full_name": "dev_sean.default.yt_channel_stats_full",
    "prefer_channel_snapshot_table": "true",
    "snapshot_channel_id_col": "canonical_id",
    "snapshot_channel_name_col": "channel_name",
    "snapshot_subscriber_col": "subscriber_count",
    "snapshot_lifetime_views_col": "total_view_count",
    "snapshot_lifetime_videos_col": "",
    "snapshot_date_col": "collected_date",
    "snapshot_timestamp_col": "collected_at",
    "videos_table": "yt_sl_videos",
    "video_metrics_table": "yt_sl_videos_metrics",
    "sampling_history_table": "sampling_history",
    "subsample_items_table": "subsample_items",
    "lid_table_full_name": "dev_sean.matt.yt_lid_v3_channels",
    "lid_run_id": "",  # blank = latest run_id in language table
    "category_source_table_full_name": "",  # optional channel-level category table override
    "category_source_channel_col": "channel_id",
    "category_source_category_col": "category",
    "backfill_categories_table_full_name": "dev_sean.default.backfill_channels",
    "prefer_backfill_topic_categories": "true",
    "category_coverage_sample_fraction": "0.001",
    "derive_category_from_videos": "false",
    "video_category_col": "ai_label",
    "category_candidate_cols": "ai_label,all_labels,label_data",
    "category_lookback_days": "0",  # 0 = no lookback filter
    # Development/discovery tables.
    "dev_catalog": "dev_sean",
    "dev_schema": "default",
    "diagnostics_schema": "diagnostics",
    "validation_schema": "validation",
    "threshold_1k_schema": "threshold_yt_1k",
    "threshold_5k_schema": "threshold_yt_5k",
    # Outputs.
    "output_catalog": "dev_sean",
    "output_schema": "matt",
    "output_table_prefix": "codex_yt_attention_ms",
    "output_base_dir": "/Volumes/dev_sean/matt/youtube_attention_manuscript/codex_work_product",
    # Measures and estimands.
    "target_capture_date": "",  # blank = auto-complete anchor selection; set manually only after confirming the partition is complete
    "anchor_selection_mode": "auto_complete",  # auto_complete, latest, manual
    "anchor_min_fraction_of_max_partition": "0.90",
    "attention_windows_days": "1,7,14,28",
    "primary_attention_window_days": "7",
    "min_primary_elapsed_days": "6",
    "allow_short_window_fallback": "false",
    "negative_delta_policy": "null_invalid",  # null_invalid, floor_zero, keep
    "observed_attention_subscriber_floor": "10000",
    "production_window_days": "28",
    "format_lookback_days": "90",
    "speaker_population_table_full_name": "",
    "shorts_max_seconds": "180",
    "traffic_blocks": "20",
    "traffic_block_value_bins": "4000",
    "thresholds_subscribers": "1000,10000,100000,1000000",
    "top_treemap_channels": "900",
    "treemap_min_channel_share": "0.00005",
    # Design-based estimation hooks. Leave blank until exact sample design is confirmed.
    "sample_design_table_full_name": "",
    "sample_weight_col": "",
    "sample_stratum_col": "",
    "sample_unit_col": "channel_id",
    "bootstrap_replicates": "300",
    "residual_tail_lower_view_share": "0.0",
    "residual_tail_upper_view_share": "",
}

for widget_name, default in WIDGET_DEFAULTS.items():
    _create_text_widget(widget_name, default)

EXECUTION_MODE = _get_widget("execution_mode", "manifest_only").strip().lower()
CONFIRM_EXPENSIVE = _get_bool_widget("confirm_expensive", False)
INSTALL_OPTIONAL_PACKAGES = _get_bool_widget("install_optional_packages", False)
RANDOM_SEED = _get_int_widget("random_seed", 20260529)
SPARK_SHUFFLE_PARTITIONS = _get_int_widget("spark_shuffle_partitions", 800)
SMOKE_CHANNEL_LIMIT = _get_int_widget("smoke_channel_limit", 50000)
WRITE_AGGREGATE_CSV = _get_bool_widget("write_aggregate_csv", True)
PERSIST_ROW_LEVEL_SCRATCH = _get_bool_widget("persist_row_level_scratch", False)
MIN_EXPORT_CELL_COUNT = _get_int_widget("min_export_cell_count", 5)
FAIL_ON_OVERSIZED_EXPORT = _get_bool_widget("fail_on_oversized_export", True)
ENABLE_DISPLAYS = _get_bool_widget("enable_displays", True)

SOURCE_CATALOG = _get_widget("source_catalog", "prod_tads")
SOURCE_SCHEMA = _get_widget("source_schema", "youtube_too")
CHANNELS_TABLE = _get_widget("channels_table", "yt_sl_channels")
CHANNEL_METRICS_TABLE = _get_widget("channel_metrics_table", "yt_sl_channels_metrics")
CHANNEL_SNAPSHOT_TABLE_FULL_NAME = _get_widget("channel_snapshot_table_full_name", "dev_sean.default.yt_channel_stats_full").strip()
PREFER_CHANNEL_SNAPSHOT_TABLE = _get_bool_widget("prefer_channel_snapshot_table", True)
SNAPSHOT_CHANNEL_ID_COL = _get_widget("snapshot_channel_id_col", "canonical_id").strip()
SNAPSHOT_CHANNEL_NAME_COL = _get_widget("snapshot_channel_name_col", "channel_name").strip()
SNAPSHOT_SUBSCRIBER_COL = _get_widget("snapshot_subscriber_col", "subscriber_count").strip()
SNAPSHOT_LIFETIME_VIEWS_COL = _get_widget("snapshot_lifetime_views_col", "total_view_count").strip()
SNAPSHOT_LIFETIME_VIDEOS_COL = _get_widget("snapshot_lifetime_videos_col", "").strip()
SNAPSHOT_DATE_COL = _get_widget("snapshot_date_col", "collected_date").strip()
SNAPSHOT_TIMESTAMP_COL = _get_widget("snapshot_timestamp_col", "collected_at").strip()
VIDEOS_TABLE = _get_widget("videos_table", "yt_sl_videos")
VIDEO_METRICS_TABLE = _get_widget("video_metrics_table", "yt_sl_videos_metrics")
SAMPLING_HISTORY_TABLE = _get_widget("sampling_history_table", "sampling_history")
SUBSAMPLE_ITEMS_TABLE = _get_widget("subsample_items_table", "subsample_items")
LID_TABLE_FULL_NAME = _get_widget("lid_table_full_name", "dev_sean.matt.yt_lid_v3_channels")
LID_RUN_ID = _get_widget("lid_run_id", "").strip()
CATEGORY_SOURCE_TABLE_FULL_NAME = _get_widget("category_source_table_full_name", "").strip()
CATEGORY_SOURCE_CHANNEL_COL = _get_widget("category_source_channel_col", "channel_id")
CATEGORY_SOURCE_CATEGORY_COL = _get_widget("category_source_category_col", "category")
BACKFILL_CATEGORIES_TABLE_FULL_NAME = _get_widget("backfill_categories_table_full_name", "dev_sean.default.backfill_channels").strip()
PREFER_BACKFILL_TOPIC_CATEGORIES = _get_bool_widget("prefer_backfill_topic_categories", True)
CATEGORY_COVERAGE_SAMPLE_FRACTION = _get_float_widget("category_coverage_sample_fraction", 0.001)
DERIVE_CATEGORY_FROM_VIDEOS = _get_bool_widget("derive_category_from_videos", True)
VIDEO_CATEGORY_COL = _get_widget("video_category_col", "ai_label")
CATEGORY_CANDIDATE_COLS = []
for _candidate in [VIDEO_CATEGORY_COL] + [c.strip() for c in _get_widget("category_candidate_cols", "ai_label,all_labels,label_data").split(",") if c.strip()]:
    if _candidate and _candidate not in CATEGORY_CANDIDATE_COLS:
        CATEGORY_CANDIDATE_COLS.append(_candidate)
CATEGORY_LOOKBACK_DAYS = _get_int_widget("category_lookback_days", 0)

DEV_CATALOG = _get_widget("dev_catalog", "dev_sean")
DEV_SCHEMA = _get_widget("dev_schema", "default")
DIAGNOSTICS_SCHEMA = _get_widget("diagnostics_schema", "diagnostics")
VALIDATION_SCHEMA = _get_widget("validation_schema", "validation")
THRESHOLD_1K_SCHEMA = _get_widget("threshold_1k_schema", "threshold_yt_1k")
THRESHOLD_5K_SCHEMA = _get_widget("threshold_5k_schema", "threshold_yt_5k")

OUTPUT_CATALOG = _get_widget("output_catalog", "dev_sean")
OUTPUT_SCHEMA = _get_widget("output_schema", "matt")
OUTPUT_TABLE_PREFIX = _safe_token(_get_widget("output_table_prefix", "codex_yt_attention_ms"), "codex_yt_attention_ms")
OUTPUT_BASE_DIR = _get_widget("output_base_dir", "/Volumes/dev_sean/matt/youtube_attention_manuscript/codex_work_product").rstrip("/")

TARGET_CAPTURE_DATE = _get_widget("target_capture_date", "").strip()
SELECTED_ATTENTION_ANCHOR_DATE = TARGET_CAPTURE_DATE
ANCHOR_SELECTION_MODE = _get_widget("anchor_selection_mode", "auto_complete").strip().lower()
ANCHOR_MIN_FRACTION_OF_MAX_PARTITION = _get_float_widget("anchor_min_fraction_of_max_partition", 0.90)
ATTENTION_WINDOWS = sorted({int(x.strip()) for x in _get_widget("attention_windows_days", WIDGET_DEFAULTS["attention_windows_days"]).split(",") if x.strip()})
PRIMARY_ATTENTION_WINDOW = _get_int_widget("primary_attention_window_days", int(WIDGET_DEFAULTS["primary_attention_window_days"]))
MIN_PRIMARY_ELAPSED_DAYS = _get_int_widget("min_primary_elapsed_days", 6)
ALLOW_SHORT_WINDOW_FALLBACK = _get_bool_widget("allow_short_window_fallback", False)
NEGATIVE_DELTA_POLICY = _get_widget("negative_delta_policy", "null_invalid").strip().lower()
OBSERVED_ATTENTION_SUBSCRIBER_FLOOR = _get_int_widget("observed_attention_subscriber_floor", 10000)
PRODUCTION_WINDOW_DAYS = _get_int_widget("production_window_days", 28)
FORMAT_LOOKBACK_DAYS = _get_int_widget("format_lookback_days", 90)
SPEAKER_POPULATION_TABLE_FULL_NAME = _get_widget("speaker_population_table_full_name", "").strip()
SHORTS_MAX_SECONDS = _get_int_widget("shorts_max_seconds", 180)
TRAFFIC_BLOCKS = _get_int_widget("traffic_blocks", 20)
TRAFFIC_BLOCK_VALUE_BINS = _get_int_widget("traffic_block_value_bins", 4000)
THRESHOLDS_SUBSCRIBERS = sorted({int(x.strip()) for x in _get_widget("thresholds_subscribers", "1000,10000,100000,1000000").split(",") if x.strip()})
TOP_TREEMAP_CHANNELS = _get_int_widget("top_treemap_channels", 900)
TREEMAP_MIN_CHANNEL_SHARE = _get_float_widget("treemap_min_channel_share", 0.00005)

SAMPLE_DESIGN_TABLE_FULL_NAME = _get_widget("sample_design_table_full_name", "").strip()
SAMPLE_WEIGHT_COL = _get_widget("sample_weight_col", "").strip()
SAMPLE_STRATUM_COL = _get_widget("sample_stratum_col", "").strip()
SAMPLE_UNIT_COL = _get_widget("sample_unit_col", "channel_id").strip()
BOOTSTRAP_REPLICATES = _get_int_widget("bootstrap_replicates", 300)
RESIDUAL_TAIL_LOWER_VIEW_SHARE = _get_float_widget("residual_tail_lower_view_share", 0.0)
RESIDUAL_TAIL_UPPER_RAW = _get_widget("residual_tail_upper_view_share", "").strip()
RESIDUAL_TAIL_UPPER_VIEW_SHARE = float(RESIDUAL_TAIL_UPPER_RAW) if RESIDUAL_TAIL_UPPER_RAW else None

RUN_ID = _safe_token(_get_widget("analysis_run_id", "") or datetime.now(timezone.utc).strftime("codex_%Y%m%d_%H%M%S"), "codex_run")

if EXECUTION_MODE not in {"manifest_only", "smoke", "core", "full"}:
    raise ValueError("execution_mode must be one of: manifest_only, smoke, core, full")
RUN_COMPUTE = EXECUTION_MODE in {"smoke", "core", "full"}
if ANCHOR_SELECTION_MODE not in {"auto_complete", "latest", "manual"}:
    raise ValueError("anchor_selection_mode must be one of: auto_complete, latest, manual")
if NEGATIVE_DELTA_POLICY not in {"null_invalid", "floor_zero", "keep"}:
    raise ValueError("negative_delta_policy must be one of: null_invalid, floor_zero, keep")
if TARGET_CAPTURE_DATE and ANCHOR_SELECTION_MODE != "manual":
    print("target_capture_date is set; treating anchor_selection_mode as manual for this run.")
    ANCHOR_SELECTION_MODE = "manual"
if EXECUTION_MODE in {"core", "full"} and not CONFIRM_EXPENSIVE:
    raise ValueError("Set confirm_expensive=true before running core or full mode.")

if INSTALL_OPTIONAL_PACKAGES:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "matplotlib>=3.8",
        "seaborn>=0.13",
        "plotly>=5.20",
        "kaleido>=0.2",
        "squarify>=0.4",
        "statsmodels>=0.14",
    ])

np.random.seed(RANDOM_SEED)
spark.conf.set("spark.sql.shuffle.partitions", str(SPARK_SHUFFLE_PARTITIONS))
print(json.dumps({
    "run_id": RUN_ID,
    "execution_mode": EXECUTION_MODE,
    "run_compute": RUN_COMPUTE,
    "source": f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}",
    "output_prefix": f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.{OUTPUT_TABLE_PREFIX}",
    "output_base_dir": OUTPUT_BASE_DIR,
    "attention_windows": ATTENTION_WINDOWS,
    "primary_attention_window": PRIMARY_ATTENTION_WINDOW,
    "min_primary_elapsed_days": MIN_PRIMARY_ELAPSED_DAYS,
    "negative_delta_policy": NEGATIVE_DELTA_POLICY,
    "allow_short_window_fallback": ALLOW_SHORT_WINDOW_FALLBACK,
    "target_capture_date": TARGET_CAPTURE_DATE or "auto",
    "anchor_selection_mode": ANCHOR_SELECTION_MODE,
    "anchor_min_fraction_of_max_partition": ANCHOR_MIN_FRACTION_OF_MAX_PARTITION,
    "observed_attention_subscriber_floor": OBSERVED_ATTENTION_SUBSCRIBER_FLOOR,
    "traffic_block_value_bins": TRAFFIC_BLOCK_VALUE_BINS,
    "preferred_channel_snapshot_table": CHANNEL_SNAPSHOT_TABLE_FULL_NAME,
    "preferred_category_table": BACKFILL_CATEGORIES_TABLE_FULL_NAME,
    "category_candidate_cols": CATEGORY_CANDIDATE_COLS,
    "derive_category_from_videos": DERIVE_CATEGORY_FROM_VIDEOS,
    "persist_row_level_scratch": PERSIST_ROW_LEVEL_SCRATCH,
    "min_export_cell_count": MIN_EXPORT_CELL_COUNT,
    "fail_on_oversized_export": FAIL_ON_OVERSIZED_EXPORT,
}, indent=2))


In [ ]:

# Stop before any table scans in manifest-only mode.
if not RUN_COMPUTE:
    msg = (
        "manifest_only mode: no table scans have run. "
        "Review widgets, then set execution_mode=smoke for a limited test or core/full with confirm_expensive=true. "
        "Downstream compute cells are also guarded with RUN_COMPUTE to prevent accidental scans in interactive runs."
    )
    print(msg)
    if _in_databricks():
        dbutils.notebook.exit(msg)
    raise SystemExit(msg)


## 1. Shared Helpers


In [ ]:
def fqtn(catalog: str, schema: str, table: str) -> str:
    return f"{catalog}.{schema}.{table}"


TABLES = {
    "channels": fqtn(SOURCE_CATALOG, SOURCE_SCHEMA, CHANNELS_TABLE),
    "channel_metrics": fqtn(SOURCE_CATALOG, SOURCE_SCHEMA, CHANNEL_METRICS_TABLE),
    "channel_snapshot_panel": CHANNEL_SNAPSHOT_TABLE_FULL_NAME,
    "videos": fqtn(SOURCE_CATALOG, SOURCE_SCHEMA, VIDEOS_TABLE),
    "video_metrics": fqtn(SOURCE_CATALOG, SOURCE_SCHEMA, VIDEO_METRICS_TABLE),
    "sampling_history": fqtn(SOURCE_CATALOG, SOURCE_SCHEMA, SAMPLING_HISTORY_TABLE),
    "subsample_items": fqtn(SOURCE_CATALOG, SOURCE_SCHEMA, SUBSAMPLE_ITEMS_TABLE),
    "lid": LID_TABLE_FULL_NAME,
    "dev_backfill_channels": BACKFILL_CATEGORIES_TABLE_FULL_NAME,
    "dev_new_channels": fqtn(DEV_CATALOG, DEV_SCHEMA, "new_channels"),
    "dev_new_threshold_channels": fqtn(DEV_CATALOG, DEV_SCHEMA, "new_threshold_channels"),
    "dev_method_summary": fqtn(DEV_CATALOG, DEV_SCHEMA, "method_summary"),
    "dev_trending_batches": fqtn(DEV_CATALOG, DEV_SCHEMA, "trending_batches"),
    "dev_pub_subs_batches": fqtn(DEV_CATALOG, DEV_SCHEMA, "pub_subs_batches"),
    "dev_pub_subs_full_pass_batches": fqtn(DEV_CATALOG, DEV_SCHEMA, "pub_subs_full_pass_batches"),
    "dev_qualified_channels": fqtn(DEV_CATALOG, DEV_SCHEMA, "qualified_channels"),
    "dev_top_227k": fqtn(DEV_CATALOG, DEV_SCHEMA, "top_227k_yt"),
    "dev_socialblade_top50k": fqtn(DEV_CATALOG, DEV_SCHEMA, "updated_sb_top50k"),
    "dev_channel_stats_full": fqtn(DEV_CATALOG, DEV_SCHEMA, "yt_channel_stats_full"),
    "diagnostics_suspicion_flags": fqtn(DEV_CATALOG, DIAGNOSTICS_SCHEMA, "too_suspicion_flags"),
    "diagnostics_rank_comparison": fqtn(DEV_CATALOG, DIAGNOSTICS_SCHEMA, "too_rank_comparison"),
    "diagnostics_run_summary": fqtn(DEV_CATALOG, DIAGNOSTICS_SCHEMA, "too_run_summary"),
    "diagnostics_breakdowns": fqtn(DEV_CATALOG, DIAGNOSTICS_SCHEMA, "too_breakdowns"),
    "validation_too_public": fqtn(DEV_CATALOG, VALIDATION_SCHEMA, "yt_too_public"),
    "validation_too_sample_cut": fqtn(DEV_CATALOG, VALIDATION_SCHEMA, "yt_too_sample_cut"),
    "threshold_1k_new_channels": fqtn(DEV_CATALOG, THRESHOLD_1K_SCHEMA, "new_channels"),
    "threshold_1k_new_threshold_channels": fqtn(DEV_CATALOG, THRESHOLD_1K_SCHEMA, "new_threshold_channels"),
    "threshold_5k_new_channels": fqtn(DEV_CATALOG, THRESHOLD_5K_SCHEMA, "new_channels"),
    "threshold_5k_new_threshold_channels": fqtn(DEV_CATALOG, THRESHOLD_5K_SCHEMA, "new_threshold_channels"),
}


def output_table(short_name: str) -> str:
    return fqtn(OUTPUT_CATALOG, OUTPUT_SCHEMA, f"{OUTPUT_TABLE_PREFIX}_{_safe_token(short_name)}")


def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).schema
        return True
    except Exception:
        return False


def read_table_optional(table_name: str) -> Optional[DataFrame]:
    try:
        return spark.table(table_name)
    except Exception as exc:
        print(f"Optional table unavailable: {table_name} ({type(exc).__name__}: {exc})")
        return None


def empty_df_from_schema(schema_spec: Dict[str, str]) -> DataFrame:
    type_map = {
        "string": StringType(),
        "date": DateType(),
        "double": DoubleType(),
        "integer": IntegerType(),
        "int": IntegerType(),
        "long": LongType(),
        "boolean": BooleanType(),
        "bool": BooleanType(),
    }
    fields = []
    for name, dtype in schema_spec.items():
        dtype_key = dtype.lower()
        if dtype_key not in type_map:
            raise ValueError(f"Unsupported empty DataFrame dtype for {name}: {dtype}")
        fields.append(StructField(name, type_map[dtype_key], True))
    return spark.createDataFrame([], StructType(fields))


def _columns_lower_map(df: DataFrame) -> Dict[str, str]:
    return {c.lower(): c for c in df.columns}


def first_existing_column(df: DataFrame, candidates: Sequence[str], required: bool = True, label: str = "column") -> Optional[str]:
    cmap = _columns_lower_map(df)
    for c in candidates:
        if c.lower() in cmap:
            return cmap[c.lower()]
    if required:
        raise ValueError(f"Could not find {label}. Tried {candidates}. Available columns: {df.columns}")
    return None


def display_if_enabled(df: DataFrame, n: int = 20) -> None:
    if ENABLE_DISPLAYS and _in_databricks():
        display(df.limit(n))
    else:
        df.show(n, truncate=False)


def ensure_output_dir(path: str) -> str:
    if path.startswith("dbfs:/"):
        if _in_databricks():
            dbutils.fs.mkdirs(path)
        return "/dbfs/" + path.replace("dbfs:/", "", 1).lstrip("/")
    Path(path).mkdir(parents=True, exist_ok=True)
    return path



FIG_DIR = ensure_output_dir(f"{OUTPUT_BASE_DIR}/{RUN_ID}/figures")
TABLE_EXPORT_DIR = ensure_output_dir(f"{OUTPUT_BASE_DIR}/{RUN_ID}/aggregate_tables")

RUN_MANIFEST: Dict[str, object] = {
    "author": "codex",
    "notebook": "codex_youtube_attention_manuscript_analysis",
    "run_id": RUN_ID,
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "execution_mode": EXECUTION_MODE,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "parameters": {
        "source": f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}",
        "channel_snapshot_table": CHANNEL_SNAPSHOT_TABLE_FULL_NAME,
        "lid_table": LID_TABLE_FULL_NAME,
        "lid_run_id": LID_RUN_ID or "latest",
        "backfill_categories_table": BACKFILL_CATEGORIES_TABLE_FULL_NAME,
        "output_prefix": f"{OUTPUT_CATALOG}.{OUTPUT_SCHEMA}.{OUTPUT_TABLE_PREFIX}",
        "attention_windows": ATTENTION_WINDOWS,
        "primary_attention_window": PRIMARY_ATTENTION_WINDOW,
        "negative_delta_policy": NEGATIVE_DELTA_POLICY,
        "derive_category_from_videos": DERIVE_CATEGORY_FROM_VIDEOS,
        "persist_row_level_scratch": PERSIST_ROW_LEVEL_SCRATCH,
        "traffic_blocks": TRAFFIC_BLOCKS,
        "traffic_block_value_bins": TRAFFIC_BLOCK_VALUE_BINS,
        "min_export_cell_count": MIN_EXPORT_CELL_COUNT,
    },
    "outputs": [],
    "warnings": [],
}


def _sha256_of_file(path: str) -> str:
    h = hashlib.sha256()
    try:
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return ""


def record_manifest_output(kind: str, name: str, path_or_table: str, **extra) -> None:
    rec = {"kind": kind, "name": name, "path_or_table": path_or_table, **extra}
    RUN_MANIFEST.setdefault("outputs", []).append(rec)  # type: ignore[union-attr]


def warn_manifest(message: str) -> None:
    print(f"WARNING: {message}")
    RUN_MANIFEST.setdefault("warnings", []).append(message)  # type: ignore[union-attr]


def write_delta(df: DataFrame, short_name: str, partition_cols: Optional[Sequence[str]] = None) -> str:
    target = output_table(short_name)
    df_out = df.withColumn("analysis_run_id", F.lit(RUN_ID)).withColumn("analysis_created_at", F.current_timestamp())
    writer = df_out.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.saveAsTable(target)
    record_manifest_output("delta_table", short_name, target)
    print(f"Wrote {target}")
    return target


def write_internal_scratch_delta(df: DataFrame, short_name: str, partition_cols: Optional[Sequence[str]] = None) -> str:
    """Persist row-level scratch only when explicitly requested.

    Aggregate manuscript outputs still use write_delta(). Row-level frames can be useful for debugging inside
    Databricks, but writing them by default creates avoidable cost and a larger data-governance surface.
    """
    target = output_table(short_name)
    if not PERSIST_ROW_LEVEL_SCRATCH:
        record_manifest_output(
            "row_level_scratch_skipped",
            short_name,
            target,
            persistence="disabled_by_default",
            reason="set persist_row_level_scratch=true to materialize this internal row-level frame",
        )
        print(f"Skipped row-level scratch Delta table {target}; set persist_row_level_scratch=true to write it.")
        return target
    return write_delta(df, short_name, partition_cols=partition_cols)


def export_aggregate_csv(df: DataFrame, short_name: str, max_rows: int = 100000) -> None:
    if not WRITE_AGGREGATE_CSV:
        return
    count_for_export = df.limit(max_rows + 1).count()
    if count_for_export > max_rows:
        msg = f"CSV export for {short_name} would contain > {max_rows:,} rows. This is not aggregate-safe."
        if FAIL_ON_OVERSIZED_EXPORT:
            raise ValueError(msg)
        print(f"Skipping {msg}")
        return
    export_df = df
    numeric_types = {"byte", "short", "integer", "long", "float", "double", "decimal"}
    count_like_cols = [
        field.name
        for field in export_df.schema.fields
        if field.dataType.typeName() in numeric_types
        and (field.name.startswith("n_") or field.name in {"channels", "n_channels", "n_videos", "sample_rows", "metric_rows", "distinct_channels"})
    ]
    if MIN_EXPORT_CELL_COUNT > 1 and count_like_cols:
        suppress_expr = None
        for col_name in count_like_cols:
            this_expr = F.col(col_name).isNotNull() & (F.col(col_name) > F.lit(0)) & (F.col(col_name) < F.lit(MIN_EXPORT_CELL_COUNT))
            suppress_expr = this_expr if suppress_expr is None else (suppress_expr | this_expr)
        suppressed = export_df.where(suppress_expr).count() if suppress_expr is not None else 0
        if suppressed:
            print(f"Suppressing {suppressed:,} small-cell rows from {short_name} CSV export (min_export_cell_count={MIN_EXPORT_CELL_COUNT}).")
            export_df = export_df.where(~suppress_expr)
    export_df = export_df.withColumn("analysis_author", F.lit("codex"))
    path = f"{TABLE_EXPORT_DIR}/{_safe_token(short_name)}.csv"
    pdf = export_df.toPandas()
    pdf.to_csv(path, index=False)
    record_manifest_output("aggregate_csv", short_name, path, rows=int(len(pdf)), sha256=_sha256_of_file(path))
    print(f"Wrote aggregate CSV {path}")

def load_output(short_name: str, run_id: Optional[str] = None) -> DataFrame:
    df = spark.table(output_table(short_name))
    chosen_run = run_id or RUN_ID
    return df.where(F.col("analysis_run_id") == chosen_run)


def parse_int_csv(raw: str) -> List[int]:
    return [int(x.strip()) for x in raw.split(",") if x.strip()]


def subscriber_band_expr(col_name: str = "subscribers"):
    c = F.col(col_name)
    return (
        F.when(c >= 10_000_000, F.lit("10M+"))
        .when(c >= 1_000_000, F.lit("1M-10M"))
        .when(c >= 100_000, F.lit("100k-1M"))
        .when(c >= 10_000, F.lit("10k-100k"))
        .when(c >= 1_000, F.lit("1k-10k"))
        .when(c >= 1, F.lit("1-1k"))
        .otherwise(F.lit("0/hidden/missing"))
    )


def view_band_expr(col_name: str = "lifetime_views"):
    c = F.col(col_name)
    return (
        F.when(c >= 10_000_000_000, F.lit("10B+"))
        .when(c >= 1_000_000_000, F.lit("1B-10B"))
        .when(c >= 100_000_000, F.lit("100M-1B"))
        .when(c >= 10_000_000, F.lit("10M-100M"))
        .when(c >= 1_000_000, F.lit("1M-10M"))
        .when(c >= 100_000, F.lit("100k-1M"))
        .when(c >= 1, F.lit("1-100k"))
        .otherwise(F.lit("0/missing"))
    )


def canonical_language_expr() -> F.Column:
    return F.coalesce(
        F.col("consensus_for_rollup_label"),
        F.col("consensus_analysis_language_cluster"),
        F.col("consensus_language_iso639_3"),
        F.col("detected_language"),
        F.col("language_code"),
        F.lit("unknown"),
    )


## 2. Metadata Inventory and Reviewer Safety Check

This cell uses metadata calls and `LIMIT 0` table checks only. It is meant to catch schema drift before a costly run.


In [ ]:
if RUN_COMPUTE:
    metadata_rows = []
    for logical_name, table_name in TABLES.items():
        exists = table_exists(table_name)
        columns = []
        if exists:
            try:
                columns = spark.table(table_name).limit(0).columns
            except Exception:
                columns = []
        metadata_rows.append({
            "logical_name": logical_name,
            "table_name": table_name,
            "exists": exists,
            "n_columns_visible": len(columns),
            "columns_preview": ", ".join(columns[:12]),
        })

    metadata_df = spark.createDataFrame(pd.DataFrame(metadata_rows))
    write_delta(metadata_df, "metadata_table_check")
    display_if_enabled(metadata_df.orderBy("logical_name"), n=100)
else:
    print('Skipped cell 10: execution_mode=manifest_only.')


## 3. Build the Channel Analysis Frame

The channel frame is the common denominator for most manuscript results. It joins current channel metrics, recent view deltas, public metadata, language labels, category labels, and optional country/diagnostic fields.

**Attention definition:** the preferred current-attention measure is a weeklyized delta in cumulative channel lifetime views between repeated `capture_date` snapshots. The current operational panel is the Top of the Ocean snapshot job, approximately channels with at least 10,000 subscribers, run on Sunday evenings. The notebook keeps all available channels in the frame, but explicitly labels whether each channel has an observed current-attention delta and whether it is inside the configured observed-attention subscriber floor.

**Past-year views:** `views_past_year` from the TOO diagnostics layer is used later as a comparison metric, not as the canonical weekly-attention measure.


In [ ]:
if RUN_COMPUTE:
    def build_channel_metrics_panel() -> DataFrame:
        """Return a normalized channel-metric panel with one row per channel/date snapshot."""
        if PREFER_CHANNEL_SNAPSHOT_TABLE and CHANNEL_SNAPSHOT_TABLE_FULL_NAME and table_exists(CHANNEL_SNAPSHOT_TABLE_FULL_NAME):
            snap = spark.table(CHANNEL_SNAPSHOT_TABLE_FULL_NAME)
            required = [SNAPSHOT_CHANNEL_ID_COL, SNAPSHOT_SUBSCRIBER_COL, SNAPSHOT_LIFETIME_VIEWS_COL, SNAPSHOT_DATE_COL]
            missing = [c for c in required if c and c not in snap.columns]
            if missing:
                raise ValueError(f"Configured channel snapshot table is missing required columns: {missing}")
            raw = snap.select(
                F.col(SNAPSHOT_CHANNEL_ID_COL).cast("string").alias("channel_id"),
                F.col(SNAPSHOT_CHANNEL_NAME_COL).cast("string").alias("channel_name") if SNAPSHOT_CHANNEL_NAME_COL in snap.columns else F.lit(None).cast("string").alias("channel_name"),
                F.col(SNAPSHOT_SUBSCRIBER_COL).cast("double").alias("subscribers"),
                F.col(SNAPSHOT_LIFETIME_VIEWS_COL).cast("double").alias("lifetime_views"),
                F.col(SNAPSHOT_LIFETIME_VIDEOS_COL).cast("double").alias("lifetime_videos") if SNAPSHOT_LIFETIME_VIDEOS_COL in snap.columns else F.lit(None).cast("double").alias("lifetime_videos"),
                F.to_date(F.col(SNAPSHOT_DATE_COL)).alias("capture_date"),
                F.col(SNAPSHOT_TIMESTAMP_COL).cast("timestamp").alias("metric_timestamp") if SNAPSHOT_TIMESTAMP_COL in snap.columns else F.lit(None).cast("timestamp").alias("metric_timestamp"),
                F.lit(CHANNEL_SNAPSHOT_TABLE_FULL_NAME).alias("channel_metric_source"),
            )
            return raw.where(F.col("channel_id").isNotNull() & F.col("capture_date").isNotNull())

        raw = spark.table(TABLES["channel_metrics"]).select(
            F.col("channel_id").cast("string"),
            F.col("channel_name").cast("string"),
            F.col("follower_count").cast("double").alias("subscribers"),
            F.col("views_count").cast("double").alias("lifetime_views"),
            F.col("post_count").cast("double").alias("lifetime_videos"),
            F.to_date("capture_date").alias("capture_date"),
            F.lit(None).cast("timestamp").alias("metric_timestamp"),
            F.lit(TABLES["channel_metrics"]).alias("channel_metric_source"),
        )
        return raw.where(F.col("channel_id").isNotNull() & F.col("capture_date").isNotNull())


    def choose_attention_anchor_date(raw_panel: DataFrame) -> Tuple[str, DataFrame]:
        """Choose a complete global anchor partition and return its date plus partition diagnostics."""
        coverage = raw_panel.groupBy("capture_date").agg(
            F.count("*").alias("metric_rows"),
            F.countDistinct("channel_id").alias("distinct_channels"),
            F.sum(F.when(F.col("subscribers") >= OBSERVED_ATTENTION_SUBSCRIBER_FLOOR, 1).otherwise(0)).alias("rows_at_or_above_attention_floor"),
        )
        max_rows = coverage.agg(F.max("metric_rows").alias("max_rows")).collect()[0]["max_rows"]
        if max_rows is None or max_rows <= 0:
            raise ValueError("No channel metric snapshot rows are available for anchor selection.")
        coverage = coverage.withColumn("row_fraction_of_max_partition", F.col("metric_rows") / F.lit(float(max_rows))).withColumn(
            "passes_auto_complete_rule", F.col("row_fraction_of_max_partition") >= F.lit(float(ANCHOR_MIN_FRACTION_OF_MAX_PARTITION))
        )
        if TARGET_CAPTURE_DATE:
            anchor_expr = F.to_date(F.lit(TARGET_CAPTURE_DATE))
            selected = TARGET_CAPTURE_DATE
            mode_used = "manual_target_capture_date"
            target_exists = coverage.where(F.col("capture_date") == anchor_expr).limit(1).count() > 0
            if not target_exists:
                raise ValueError(f"Configured target_capture_date={TARGET_CAPTURE_DATE} does not exist in the channel metric panel.")
        elif ANCHOR_SELECTION_MODE == "latest":
            selected = str(coverage.agg(F.max("capture_date").alias("capture_date")).collect()[0]["capture_date"])
            mode_used = "latest_visible_partition"
        else:
            row = coverage.where(F.col("passes_auto_complete_rule")).agg(F.max("capture_date").alias("capture_date")).collect()[0]
            if not row["capture_date"]:
                raise ValueError("No anchor partition passes anchor_min_fraction_of_max_partition; inspect attention_anchor_snapshot_coverage.")
            selected = str(row["capture_date"])
            mode_used = "latest_complete_partition"
        coverage = coverage.withColumn("selected_anchor_date", F.lit(selected)).withColumn(
            "anchor_selection_mode_used", F.lit(mode_used)
        ).withColumn("is_selected_anchor", F.col("capture_date") == F.to_date(F.lit(selected))).orderBy("capture_date")
        return selected, coverage


    def build_latest_channel_metrics() -> DataFrame:
        global SELECTED_ATTENTION_ANCHOR_DATE
        raw = build_channel_metrics_panel()

        anchor_date, anchor_coverage = choose_attention_anchor_date(raw)
        SELECTED_ATTENTION_ANCHOR_DATE = anchor_date
        write_delta(anchor_coverage, "attention_anchor_snapshot_coverage")
        export_aggregate_csv(anchor_coverage, "attention_anchor_snapshot_coverage")
        print(f"Using attention anchor date {anchor_date}.")

        complete_prior_dates = anchor_coverage.where(
            F.col("passes_auto_complete_rule") & (F.col("capture_date") < F.to_date(F.lit(anchor_date)))
        ).select(F.col("capture_date").alias("complete_prior_capture_date"))
        prior_partition_status = anchor_coverage.where(
            F.col("capture_date") < F.to_date(F.lit(anchor_date))
        ).select(
            F.col("capture_date").alias("prior_capture_date"),
            "metric_rows",
            "distinct_channels",
            "rows_at_or_above_attention_floor",
            "row_fraction_of_max_partition",
            "passes_auto_complete_rule",
        ).withColumn(
            "eligible_as_prior_snapshot", F.col("passes_auto_complete_rule")
        ).withColumn(
            "anchor_date", F.to_date(F.lit(anchor_date))
        ).withColumn(
            "prior_selection_rule",
            F.lit(f"row_fraction_of_max_partition >= {ANCHOR_MIN_FRACTION_OF_MAX_PARTITION}"),
        )
        write_delta(prior_partition_status, "attention_prior_snapshot_status")
        export_aggregate_csv(prior_partition_status, "attention_prior_snapshot_status")

        if EXECUTION_MODE == "smoke":
            smoke_limit = max(int(SMOKE_CHANNEL_LIMIT), 1)
            anchor_candidates = raw.where(
                (F.col("capture_date") == F.to_date(F.lit(anchor_date)))
                & F.col("channel_id").isNotNull()
            )
            floor_ids = (
                anchor_candidates
                .where(F.col("subscribers") >= F.lit(float(OBSERVED_ATTENTION_SUBSCRIBER_FLOOR)))
                .select("channel_id")
                .dropDuplicates()
                .orderBy("channel_id")
                .limit(smoke_limit)
                .cache()
            )
            smoke_ids = floor_ids
            smoke_scope = "anchor_partition_at_or_above_observed_floor"
            smoke_n = smoke_ids.count()
            if smoke_n == 0:
                smoke_ids = (
                    anchor_candidates
                    .select("channel_id")
                    .dropDuplicates()
                    .orderBy("channel_id")
                    .limit(smoke_limit)
                    .cache()
                )
                smoke_scope = "anchor_partition_all_channels_fallback"
                smoke_n = smoke_ids.count()
            smoke_status = spark.createDataFrame(pd.DataFrame([{
                "execution_mode": EXECUTION_MODE,
                "anchor_date": anchor_date,
                "requested_smoke_channel_limit": smoke_limit,
                "sampled_channel_ids": int(smoke_n),
                "sample_scope": smoke_scope,
                "detail": "Smoke mode restricts the panel before weekly-delta self-joins and ranking; anchor and prior partition coverage are still computed before sampling."
            }]))
            write_delta(smoke_status, "smoke_channel_sample_status")
            export_aggregate_csv(smoke_status, "smoke_channel_sample_status")
            raw = raw.join(F.broadcast(smoke_ids), "channel_id", "inner")
            print(f"SMOKE: restricted channel metric panel to {smoke_n:,} sampled anchor channels before self-joins.")

        daily = raw.groupBy("channel_id", "capture_date").agg(
            F.max("channel_name").alias("channel_name_metrics"),
            F.max("subscribers").alias("subscribers"),
            F.max("lifetime_views").alias("lifetime_views"),
            F.max("lifetime_videos").alias("lifetime_videos"),
            F.max("metric_timestamp").alias("metric_timestamp"),
            F.max("channel_metric_source").alias("channel_metric_source"),
        )

        anchor = daily.where(F.col("capture_date") == F.to_date(F.lit(anchor_date))).select(
            "channel_id",
            "channel_name_metrics",
            "subscribers",
            "lifetime_views",
            "lifetime_videos",
            "metric_timestamp",
            "channel_metric_source",
            F.col("capture_date").alias("latest_capture_date"),
        )

        for window_days in ATTENTION_WINDOWS:
            prior_candidates = (
                daily.alias("m")
                .join(anchor.select("channel_id", "latest_capture_date").alias("a"), "channel_id", "inner")
                .join(
                    F.broadcast(complete_prior_dates).alias("complete_dates"),
                    F.col("m.capture_date") == F.col("complete_dates.complete_prior_capture_date"),
                    "inner",
                )
                .where(F.col("m.capture_date") <= F.date_sub(F.col("a.latest_capture_date"), window_days))
            )
            w_prior = Window.partitionBy("channel_id").orderBy(F.col("m.capture_date").desc())
            prior = prior_candidates.withColumn("rn", F.row_number().over(w_prior)).where(F.col("rn") == 1).select(
                F.col("channel_id"),
                F.col("m.capture_date").alias(f"prior_capture_date_{window_days}d"),
                F.col("m.lifetime_views").alias(f"prior_lifetime_views_{window_days}d"),
            )
            anchor = anchor.join(prior, "channel_id", "left")
            elapsed_days = F.datediff(F.col("latest_capture_date"), F.col(f"prior_capture_date_{window_days}d"))
            delta_views = F.col("lifetime_views") - F.col(f"prior_lifetime_views_{window_days}d")
            valid_delta = (elapsed_days > 0) & delta_views.isNotNull()
            weekly_views_expr = F.when(
                valid_delta,
                F.when(delta_views >= 0, delta_views / elapsed_days * F.lit(7.0))
                .when(F.lit(NEGATIVE_DELTA_POLICY == "floor_zero"), F.lit(0.0))
                .when(F.lit(NEGATIVE_DELTA_POLICY == "keep"), delta_views / elapsed_days * F.lit(7.0))
                .otherwise(F.lit(None).cast("double")),
            ).otherwise(F.lit(None).cast("double"))
            anchor = anchor.withColumn(f"elapsed_days_{window_days}d", elapsed_days).withColumn(
                f"weekly_views_{window_days}d",
                weekly_views_expr,
            ).withColumn(
                f"negative_delta_{window_days}d",
                F.when((elapsed_days > 0) & (delta_views < 0), F.lit(True)).otherwise(F.lit(False)),
            )

        preferred_windows = [PRIMARY_ATTENTION_WINDOW] + [w for w in sorted(ATTENTION_WINDOWS, reverse=True) if w != PRIMARY_ATTENTION_WINDOW]
        short_window_permitted = ALLOW_SHORT_WINDOW_FALLBACK or EXECUTION_MODE == "smoke"
        eligible_windows = [w for w in preferred_windows if short_window_permitted or w >= MIN_PRIMARY_ELAPSED_DAYS]
        if not eligible_windows:
            raise ValueError("No eligible attention windows remain after applying min_primary_elapsed_days and allow_short_window_fallback.")

        anchor = anchor.withColumn("weekly_views_primary", F.lit(None).cast("double"))
        anchor = anchor.withColumn("attention_window_used_days", F.lit(None).cast("int"))
        anchor = anchor.withColumn("attention_elapsed_days", F.lit(None).cast("int"))
        for window_days in eligible_windows:
            value_col = f"weekly_views_{window_days}d"
            elapsed_col = f"elapsed_days_{window_days}d"
            if value_col not in anchor.columns:
                continue
            use_this = F.col("weekly_views_primary").isNull() & F.col(value_col).isNotNull()
            anchor = anchor.withColumn("weekly_views_primary", F.when(use_this, F.col(value_col)).otherwise(F.col("weekly_views_primary")))
            anchor = anchor.withColumn("attention_window_used_days", F.when(use_this, F.lit(window_days)).otherwise(F.col("attention_window_used_days")))
            anchor = anchor.withColumn("attention_elapsed_days", F.when(use_this, F.col(elapsed_col).cast("int")).otherwise(F.col("attention_elapsed_days")))

        anchor = anchor.withColumn(
            "attention_measure_status",
            F.when(F.col("weekly_views_primary").isNull(), F.lit("no_valid_prior_snapshot"))
            .when(F.col("attention_window_used_days") == F.lit(PRIMARY_ATTENTION_WINDOW), F.lit("primary_window_available"))
            .when((F.col("attention_window_used_days") < F.lit(MIN_PRIMARY_ELAPSED_DAYS)) & F.lit(EXECUTION_MODE == "smoke") & F.lit(not ALLOW_SHORT_WINDOW_FALLBACK), F.lit("short_window_fallback_smoke_only"))
            .when(F.col("attention_window_used_days") < F.lit(MIN_PRIMARY_ELAPSED_DAYS), F.lit("short_window_fallback"))
            .otherwise(F.lit("longer_window_fallback")),
        ).withColumn("attention_anchor_date", F.to_date(F.lit(anchor_date))).withColumn(
            "attention_is_short_window_fallback",
            F.col("attention_window_used_days").isNotNull() & (F.col("attention_window_used_days") < F.lit(MIN_PRIMARY_ELAPSED_DAYS)),
        )

        window_usage = anchor.groupBy("attention_measure_status", "attention_window_used_days", "attention_elapsed_days").agg(
            F.count("*").alias("n_channels"),
            F.sum("weekly_views_primary").alias("weekly_views"),
            F.sum(F.when(F.col("subscribers") >= OBSERVED_ATTENTION_SUBSCRIBER_FLOOR, 1).otherwise(0)).alias("n_channels_at_or_above_attention_floor"),
        ).orderBy("attention_measure_status", "attention_window_used_days", "attention_elapsed_days")
        write_delta(window_usage, "attention_window_usage_summary")
        export_aggregate_csv(window_usage, "attention_window_usage_summary")

        return anchor.withColumn("subscriber_band", subscriber_band_expr("subscribers")).withColumn("lifetime_view_band", view_band_expr("lifetime_views"))

    def build_channel_metadata() -> DataFrame:
        ch = spark.table(TABLES["channels"]).select(
            F.col("channel_id").cast("string"),
            F.col("channel_name").cast("string").alias("channel_name_meta"),
            F.col("channel_url").cast("string"),
            F.col("channel_url_external").cast("string"),
            F.col("language_code").cast("string"),
            F.col("detected_language").cast("string"),
            F.col("first_ingestion_timestamp"),
            F.col("first_capture_timestamp"),
            F.to_date("capture_date").alias("channel_capture_date"),
        ).where(F.col("channel_id").isNotNull())
        w = Window.partitionBy("channel_id").orderBy(F.col("channel_capture_date").desc_nulls_last())
        return ch.withColumn("rn", F.row_number().over(w)).where(F.col("rn") == 1).drop("rn")


    def build_country_lookup() -> DataFrame:
        sources = []
        for logical_name, id_col, country_col in [
            ("dev_qualified_channels", "channel_id", "country"),
            ("dev_new_channels", "canonical_id", "country"),
            ("dev_new_threshold_channels", "canonical_id", "country"),
            ("threshold_1k_new_channels", "canonical_id", "country"),
            ("threshold_5k_new_channels", "canonical_id", "country"),
        ]:
            df = read_table_optional(TABLES[logical_name])
            if df is not None and id_col in df.columns and country_col in df.columns:
                sources.append(df.select(F.col(id_col).cast("string").alias("channel_id"), F.col(country_col).cast("string").alias("country")))
        if not sources:
            return empty_df_from_schema({"channel_id": "string", "country": "string"})
        out = sources[0]
        for df in sources[1:]:
            out = out.unionByName(df, allowMissingColumns=True)
        return out.where(F.col("channel_id").isNotNull()).groupBy("channel_id").agg(F.first("country", ignorenulls=True).alias("country"))


    def build_language_labels() -> DataFrame:
        lid = read_table_optional(TABLES["lid"])
        required_schema = {
            "channel_id": "string",
            "consensus_for_rollup_label": "string",
            "consensus_analysis_language_cluster": "string",
            "consensus_language_iso639_3": "string",
            "consensus_language_script": "string",
            "primary_language_confidence": "double",
            "consensus_status": "string",
            "requires_manual_adjudication": "boolean",
            "lid_run_id_used": "string",
        }
        if lid is None:
            return empty_df_from_schema(required_schema)
        run_id = LID_RUN_ID
        if not run_id and "run_id" in lid.columns:
            run_id = lid.agg(F.max("run_id").alias("run_id")).collect()[0]["run_id"]
            print(f"Using latest language run_id: {run_id}")
        if run_id and "run_id" in lid.columns:
            lid = lid.where(F.col("run_id") == run_id)
        exprs = []
        for col_name, dtype in required_schema.items():
            source_col = "run_id" if col_name == "lid_run_id_used" else col_name
            if source_col in lid.columns:
                exprs.append(F.col(source_col).cast(dtype).alias(col_name))
            else:
                exprs.append(F.lit(None).cast(dtype).alias(col_name))
        return lid.select(*exprs).where(F.col("channel_id").isNotNull()).dropDuplicates(["channel_id"])
else:
    print('Skipped cell 12: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    CATEGORY_SENTINELS = {"", "unknown", "uncategorized", "pending", "none", "null", "nan", "n/a", "na"}
    COMPLETED_BACKFILL_STATUSES = {"done", "complete", "completed", "success", "succeeded"}


    def normalize_topic_category_expr(topic_col: F.Column) -> F.Column:
        cleaned = F.regexp_replace(topic_col.cast("string"), r"^https?://[^/]+/wiki/", "")
        cleaned = F.regexp_replace(cleaned, "_", " ")
        cleaned = F.regexp_replace(cleaned, "%26", "&")
        return cleaned


    def build_backfill_topic_categories() -> DataFrame:
        if not (PREFER_BACKFILL_TOPIC_CATEGORIES and BACKFILL_CATEGORIES_TABLE_FULL_NAME and table_exists(BACKFILL_CATEGORIES_TABLE_FULL_NAME)):
            return empty_df_from_schema({"channel_id": "string", "analysis_category": "string", "category_source": "string", "topic_categories_count": "integer", "backfill_status": "string"})
        bf = spark.table(BACKFILL_CATEGORIES_TABLE_FULL_NAME)
        if "canonical_id" not in bf.columns or "topic_categories" not in bf.columns:
            return empty_df_from_schema({"channel_id": "string", "analysis_category": "string", "category_source": "string", "topic_categories_count": "integer", "backfill_status": "string"})
        status_col = F.col("status").cast("string") if "status" in bf.columns else F.lit(None).cast("string")
        status_ok = F.lower(F.trim(F.col("backfill_status"))).isin(*sorted(COMPLETED_BACKFILL_STATUSES)) if "status" in bf.columns else F.lit(True)
        return bf.select(
            F.col("canonical_id").cast("string").alias("channel_id"),
            normalize_topic_category_expr(F.element_at(F.col("topic_categories"), 1)).alias("analysis_category"),
            F.size(F.col("topic_categories")).cast("int").alias("topic_categories_count"),
            status_col.alias("backfill_status"),
            F.lit("backfill_channels.topic_categories:first_element").alias("category_source"),
        ).where(
            F.col("channel_id").isNotNull()
            & F.col("analysis_category").isNotNull()
            & (F.length(F.trim(F.col("analysis_category"))) > 0)
            & ~F.lower(F.trim(F.col("analysis_category"))).isin(*sorted(CATEGORY_SENTINELS))
            & status_ok
        ).dropDuplicates(["channel_id"])


    def build_video_derived_categories(channel_ids: Optional[DataFrame] = None) -> DataFrame:
        if not DERIVE_CATEGORY_FROM_VIDEOS:
            return empty_df_from_schema({"channel_id": "string", "analysis_category": "string", "category_source": "string", "category_video_count": "long", "category_video_views_latest": "double"})

        videos_raw = spark.table(TABLES["videos"])
        category_col = first_existing_column(videos_raw, CATEGORY_CANDIDATE_COLS, required=False, label="video category")
        if category_col is None:
            print(f"None of the configured category columns were found: {CATEGORY_CANDIDATE_COLS}. Category will be unknown.")
            return empty_df_from_schema({"channel_id": "string", "analysis_category": "string", "category_source": "string", "category_video_count": "long", "category_video_views_latest": "double"})

        v = videos_raw.select(
            F.col("channel_id").cast("string"),
            F.col("video_id").cast("string"),
            F.col(category_col).cast("string").alias("video_category"),
            F.to_date("published_date").alias("published_date"),
        ).where(
            F.col("channel_id").isNotNull()
            & F.col("video_id").isNotNull()
            & F.col("video_category").isNotNull()
            & (F.length(F.trim(F.col("video_category"))) > 0)
            & ~F.lower(F.trim(F.col("video_category"))).isin(*sorted(CATEGORY_SENTINELS))
        )
        category_anchor_date = SELECTED_ATTENTION_ANCHOR_DATE or TARGET_CAPTURE_DATE
        if category_anchor_date and CATEGORY_LOOKBACK_DAYS > 0:
            v = v.where(F.col("published_date") >= F.date_sub(F.to_date(F.lit(category_anchor_date)), CATEGORY_LOOKBACK_DAYS))
        if channel_ids is not None:
            scoped_ids = channel_ids.select("channel_id").dropDuplicates()
            if EXECUTION_MODE == "smoke":
                scoped_ids = F.broadcast(scoped_ids.limit(SMOKE_CHANNEL_LIMIT))
            v = v.join(scoped_ids, "channel_id", "inner")

        if v.limit(1).count() == 0:
            warn_manifest(
                f"Video-derived categories skipped: configured column {category_col} has no usable non-null values "
                "after source filters. Backfill/topic categories remain the default category source."
            )
            return empty_df_from_schema({"channel_id": "string", "analysis_category": "string", "category_source": "string", "category_video_count": "long", "category_video_views_latest": "double"})

        vm = spark.table(TABLES["video_metrics"]).select(
            F.col("channel_id").cast("string"),
            F.col("video_id").cast("string"),
            F.col("view_count").cast("double"),
            F.to_date("capture_date").alias("capture_date"),
        ).where(F.col("channel_id").isNotNull() & F.col("video_id").isNotNull())
        if category_anchor_date:
            vm = vm.where(F.col("capture_date") <= F.to_date(F.lit(category_anchor_date)))
        if channel_ids is not None:
            scoped_ids = channel_ids.select("channel_id").dropDuplicates()
            if EXECUTION_MODE == "smoke":
                scoped_ids = F.broadcast(scoped_ids.limit(SMOKE_CHANNEL_LIMIT))
            vm = vm.join(scoped_ids, "channel_id", "inner")
        w = Window.partitionBy("channel_id", "video_id").orderBy(F.col("capture_date").desc_nulls_last())
        vm_latest = vm.withColumn("rn", F.row_number().over(w)).where(F.col("rn") == 1).drop("rn", "capture_date")

        by_cat = v.dropDuplicates(["channel_id", "video_id", "video_category"]).join(
            vm_latest, ["channel_id", "video_id"], "left"
        ).groupBy("channel_id", "video_category").agg(
            F.countDistinct("video_id").alias("category_video_count"),
            F.sum(F.coalesce(F.col("view_count"), F.lit(0.0))).alias("category_video_views_latest"),
        )
        w_cat = Window.partitionBy("channel_id").orderBy(F.col("category_video_views_latest").desc(), F.col("category_video_count").desc(), F.col("video_category"))
        return by_cat.withColumn("rn", F.row_number().over(w_cat)).where(F.col("rn") == 1).select(
            "channel_id",
            F.col("video_category").alias("analysis_category"),
            "category_video_count",
            "category_video_views_latest",
            F.lit(f"modal_video_category_weighted_by_video_views:{category_col}").alias("category_source"),
        )


    def build_category_by_channel(channel_ids: Optional[DataFrame] = None) -> DataFrame:
        if CATEGORY_SOURCE_TABLE_FULL_NAME:
            cat = spark.table(CATEGORY_SOURCE_TABLE_FULL_NAME).select(
                F.col(CATEGORY_SOURCE_CHANNEL_COL).cast("string").alias("channel_id"),
                F.col(CATEGORY_SOURCE_CATEGORY_COL).cast("string").alias("analysis_category"),
                F.lit(CATEGORY_SOURCE_TABLE_FULL_NAME).alias("category_source"),
            )
            return cat.where(
                F.col("channel_id").isNotNull()
                & F.col("analysis_category").isNotNull()
                & (F.length(F.trim(F.col("analysis_category"))) > 0)
                & ~F.lower(F.trim(F.col("analysis_category"))).isin(*sorted(CATEGORY_SENTINELS))
            ).dropDuplicates(["channel_id"])

        topic = build_backfill_topic_categories()
        if not DERIVE_CATEGORY_FROM_VIDEOS:
            return topic
        video = build_video_derived_categories(channel_ids)
        return topic.alias("t").join(video.alias("v"), "channel_id", "full").select(
            "channel_id",
            F.coalesce(F.col("t.analysis_category"), F.col("v.analysis_category")).alias("analysis_category"),
            F.coalesce(F.col("t.category_source"), F.col("v.category_source")).alias("category_source"),
            F.col("t.topic_categories_count"),
            F.col("t.backfill_status"),
            F.col("v.category_video_count"),
            F.col("v.category_video_views_latest"),
        )


    channel_metrics = build_latest_channel_metrics()
    if EXECUTION_MODE == "smoke":
        channel_metrics_for_category = channel_metrics.orderBy(F.col("weekly_views_primary").desc_nulls_last()).limit(SMOKE_CHANNEL_LIMIT)
    else:
        channel_metrics_for_category = channel_metrics.select("channel_id")

    channel_frame = (
        channel_metrics
        .join(build_channel_metadata(), "channel_id", "left")
        .join(build_language_labels(), "channel_id", "left")
        .join(build_country_lookup(), "channel_id", "left")
        .join(build_category_by_channel(channel_metrics_for_category), "channel_id", "left")
        .withColumn("channel_name", F.coalesce(F.col("channel_name_meta"), F.col("channel_name_metrics")))
        .withColumn("analysis_language", canonical_language_expr())
        .withColumn("analysis_category", F.coalesce(F.col("analysis_category"), F.lit("unknown")))
        .withColumn("views_per_subscriber_primary", F.when(F.col("subscribers") > 0, F.col("weekly_views_primary") / F.col("subscribers")))
        .withColumn(
            "attention_observation_scope",
            F.when(
                F.col("weekly_views_primary").isNotNull() & (F.col("subscribers") >= OBSERVED_ATTENTION_SUBSCRIBER_FLOOR),
                F.lit("observed_weekly_delta_top_of_ocean_floor"),
            )
            .when(F.col("weekly_views_primary").isNotNull(), F.lit("observed_weekly_delta_other"))
            .otherwise(F.lit("no_current_delta_available")),
        )
    )

    if EXECUTION_MODE == "smoke":
        channel_frame = channel_frame.orderBy(F.col("weekly_views_primary").desc_nulls_last()).limit(SMOKE_CHANNEL_LIMIT)

    write_internal_scratch_delta(channel_frame, "channel_analysis_frame")
    quality_summary = channel_frame.agg(
        F.count("*").alias("n_channels"),
        F.sum(F.when(F.col("weekly_views_primary").isNotNull(), 1).otherwise(0)).alias("n_with_current_attention"),
        F.sum(F.when((F.col("weekly_views_primary").isNotNull()) & (F.col("subscribers") >= OBSERVED_ATTENTION_SUBSCRIBER_FLOOR), 1).otherwise(0)).alias("n_with_current_attention_at_or_above_floor"),
        F.sum(F.when(F.col("subscribers").isNotNull(), 1).otherwise(0)).alias("n_with_subscribers"),
        F.sum(F.when(F.col("analysis_language") == "unknown", 1).otherwise(0)).alias("n_unknown_language"),
        F.sum(F.when(F.col("analysis_category") == "unknown", 1).otherwise(0)).alias("n_unknown_category"),
        F.sum(F.coalesce(F.col("weekly_views_primary"), F.lit(0.0))).alias("weekly_views_primary_sum"),
    )
    write_delta(quality_summary, "channel_frame_quality_summary")

    attention_scope_summary = channel_frame.groupBy("subscriber_band", "attention_observation_scope").agg(
        F.count("*").alias("n_channels"),
        F.sum(F.coalesce(F.col("weekly_views_primary"), F.lit(0.0))).alias("weekly_views_primary"),
        F.sum(F.coalesce(F.col("lifetime_views"), F.lit(0.0))).alias("lifetime_views"),
    ).orderBy("subscriber_band", "attention_observation_scope")
    write_delta(attention_scope_summary, "attention_observation_scope_summary")
    export_aggregate_csv(attention_scope_summary, "attention_observation_scope_summary")

    display_if_enabled(quality_summary)
else:
    print('Skipped cell 13: execution_mode=manifest_only.')


## 3a. Snapshot Panel Coverage

These aggregates verify whether the repeated-capture panel needed for weeklyized attention exists, how deep it is, and where coverage falls across subscriber strata.


In [ ]:
if RUN_COMPUTE:
    channel_metrics_raw = build_channel_metrics_panel().select(
        "channel_id",
        F.col("subscribers").alias("subscribers_snapshot"),
        F.col("lifetime_views").alias("lifetime_views_snapshot"),
        "capture_date",
        "channel_metric_source",
    )

    snapshot_date_summary = channel_metrics_raw.groupBy("channel_metric_source", "capture_date").agg(
        F.count("*").alias("metric_rows"),
        F.countDistinct("channel_id").alias("distinct_channels"),
        F.sum(F.when(F.col("subscribers_snapshot") >= OBSERVED_ATTENTION_SUBSCRIBER_FLOOR, 1).otherwise(0)).alias("rows_at_or_above_attention_floor"),
    ).orderBy("channel_metric_source", "capture_date")
    write_delta(snapshot_date_summary, "attention_snapshot_date_summary")
    export_aggregate_csv(snapshot_date_summary, "attention_snapshot_date_summary")

    channel_panel_depth = channel_metrics_raw.groupBy("channel_id").agg(
        F.countDistinct("capture_date").alias("n_capture_dates"),
        F.min("capture_date").alias("first_metric_capture_date"),
        F.max("capture_date").alias("last_metric_capture_date"),
        F.max("subscribers_snapshot").alias("max_subscribers_observed"),
    ).withColumn("panel_span_days", F.datediff(F.col("last_metric_capture_date"), F.col("first_metric_capture_date"))).withColumn(
        "capture_depth_bin",
        F.when(F.col("n_capture_dates") >= 12, F.lit("12+"))
        .when(F.col("n_capture_dates") >= 8, F.lit("8-11"))
        .when(F.col("n_capture_dates") >= 4, F.lit("4-7"))
        .when(F.col("n_capture_dates") >= 2, F.lit("2-3"))
        .otherwise(F.lit("1")),
    ).withColumn("subscriber_band", subscriber_band_expr("max_subscribers_observed"))

    panel_depth_summary = channel_panel_depth.groupBy("subscriber_band", "capture_depth_bin").agg(
        F.count("*").alias("n_channels"),
        F.expr("percentile_approx(panel_span_days, 0.5, 1000)").alias("median_panel_span_days"),
    ).orderBy("subscriber_band", "capture_depth_bin")
    write_delta(panel_depth_summary, "attention_panel_depth_summary")
    export_aggregate_csv(panel_depth_summary, "attention_panel_depth_summary")
    display_if_enabled(snapshot_date_summary)
    display_if_enabled(panel_depth_summary)
else:
    print('Skipped cell 15: execution_mode=manifest_only.')


## 3b. Anchor-Week and Category-Coverage Probes

These aggregate checks help choose the manuscript anchor date and quantify how complete `backfill_channels.topic_categories` is before relying on it for Figure 2.


In [ ]:
if RUN_COMPUTE:
    # Anchor-date candidates from discovery logs and the preferred channel snapshot panel.
    anchor_rows = []
    for logical_name, date_col, label in [
        ("dev_trending_batches", "run_date", "trending_batches"),
        ("dev_pub_subs_batches", "run_date", "pub_subs_batches"),
        ("dev_pub_subs_full_pass_batches", "run_date", "pub_subs_full_pass_batches"),
    ]:
        df = read_table_optional(TABLES[logical_name])
        if df is not None and date_col in df.columns:
            row = df.agg(F.max(F.to_date(F.col(date_col))).alias("max_date")).collect()[0]
            anchor_rows.append({"source": label, "max_date": str(row["max_date"]) if row["max_date"] else None})

    panel = read_table_optional(TABLES["channel_snapshot_panel"])
    if panel is not None and SNAPSHOT_DATE_COL in panel.columns:
        row = panel.agg(F.max(F.to_date(F.col(SNAPSHOT_DATE_COL))).alias("max_date")).collect()[0]
        anchor_rows.append({"source": "preferred_channel_snapshot_panel", "max_date": str(row["max_date"]) if row["max_date"] else None})

    anchor_df = spark.createDataFrame(pd.DataFrame(anchor_rows)) if anchor_rows else empty_df_from_schema({"source": "string", "max_date": "string"})
    write_delta(anchor_df, "anchor_date_candidates")
    export_aggregate_csv(anchor_df, "anchor_date_candidates")

    # Category coverage. In smoke mode use a sampled estimate; in core/full this exact aggregate is still row-level-safe but can scan backfill_channels.
    bf = read_table_optional(TABLES["dev_backfill_channels"])
    if bf is not None and "topic_categories" in bf.columns:
        bf_cov = bf
        coverage_mode = "exact"
        if EXECUTION_MODE == "smoke" and CATEGORY_COVERAGE_SAMPLE_FRACTION > 0:
            bf_cov = bf_cov.sample(False, CATEGORY_COVERAGE_SAMPLE_FRACTION, RANDOM_SEED)
            coverage_mode = f"sample_{CATEGORY_COVERAGE_SAMPLE_FRACTION}"
        status_expr = F.col("status").cast("string") if "status" in bf_cov.columns else F.lit("unknown")
        category_coverage = bf_cov.groupBy(status_expr.alias("backfill_status")).agg(
            F.count("*").alias("n_rows"),
            F.avg(F.when(F.col("topic_categories").isNotNull() & (F.size(F.col("topic_categories")) > 0), F.lit(1.0)).otherwise(F.lit(0.0))).alias("topic_category_coverage"),
            F.max(F.col("backfilled_at")).alias("max_backfilled_at") if "backfilled_at" in bf_cov.columns else F.lit(None).cast("timestamp").alias("max_backfilled_at"),
        ).withColumn("coverage_mode", F.lit(coverage_mode))
    else:
        category_coverage = spark.createDataFrame(pd.DataFrame([{"backfill_status": "unavailable", "n_rows": None, "topic_category_coverage": None, "max_backfilled_at": None, "coverage_mode": "unavailable"}]))
    write_delta(category_coverage, "backfill_topic_category_coverage")
    export_aggregate_csv(category_coverage, "backfill_topic_category_coverage")
    display_if_enabled(anchor_df)
    display_if_enabled(category_coverage, n=50)
else:
    print('Skipped cell 17: execution_mode=manifest_only.')


## 4. Attention Ranking and Traffic Blocks

Traffic blocks split the attention distribution into equal shares of current weekly views. With `traffic_blocks=20`, each block contains approximately 5% of measured current attention, ordered from highest-attention channels to lowest.


In [ ]:
if RUN_COMPUTE:

    def add_attention_rank_blocks(df: DataFrame, attention_col: str = "weekly_views_primary", n_blocks: int = TRAFFIC_BLOCKS) -> DataFrame:
        """Assign approximate equal-attention blocks without a global Spark orderBy window.

        The previous implementation used an exact unpartitioned Window.orderBy over all positive-attention
        channels. That is precise for modest frames but fragile at platform scale. This version follows the
        safer Claude design: aggregate positive values into log-spaced bins, compute the cumulative attention
        distribution on the tiny bin table, then broadcast the bin->block map back to channels. Rank fields are
        approximate bin ranges, retained only for compatibility with downstream aggregate diagnostics.
        """
        base = df.where(F.col(attention_col).isNotNull() & (F.col(attention_col) > 0))
        stats = base.withColumn("_log10_attention", F.log10(F.col(attention_col))).agg(
            F.count("*").alias("positive_channels"),
            F.sum(attention_col).alias("total_attention"),
            F.min("_log10_attention").alias("min_log10_attention"),
            F.max("_log10_attention").alias("max_log10_attention"),
        ).collect()[0]
        total_attention = stats["total_attention"]
        positive_channels = int(stats["positive_channels"] or 0)
        if total_attention is None or total_attention <= 0 or positive_channels == 0:
            msg = (
                "No positive primary attention values are available for rank/block construction. "
                "This is expected when the visible panel lacks a valid >=6-day prior snapshot and "
                "allow_short_window_fallback=false. Use execution_mode=smoke for a one-day diagnostic "
                "dry run, or wait for a complete weekly pair for manuscript estimates."
            )
            status = spark.createDataFrame(pd.DataFrame([{
                "component": "attention_rank_blocks",
                "status": "not_computed_no_positive_primary_attention",
                "detail": msg,
                "primary_attention_window_days": PRIMARY_ATTENTION_WINDOW,
                "min_primary_elapsed_days": MIN_PRIMARY_ELAPSED_DAYS,
                "allow_short_window_fallback": ALLOW_SHORT_WINDOW_FALLBACK,
                "execution_mode": EXECUTION_MODE,
            }]))
            write_delta(status, "attention_rank_status")
            export_aggregate_csv(status, "attention_rank_status")
            empty_mapping = empty_df_from_schema({
                "attention_value_bin": "integer",
                "traffic_block": "integer",
                "approx_attention_rank_lower": "integer",
                "approx_attention_rank_upper": "integer",
                "n_channels_in_bin": "long",
                "attention_in_bin": "double",
                "min_attention_in_bin": "double",
                "max_attention_in_bin": "double",
            })
            write_delta(empty_mapping, "attention_value_bin_map")
            export_aggregate_csv(empty_mapping, "attention_value_bin_map")
            warn_manifest(msg)
            return (
                df.where(F.lit(False))
                .withColumn("attention_value_bin", F.lit(None).cast("int"))
                .withColumn("traffic_block", F.lit(None).cast("int"))
                .withColumn("approx_attention_rank_lower", F.lit(None).cast("int"))
                .withColumn("approx_attention_rank_upper", F.lit(None).cast("int"))
                .withColumn("n_channels_in_bin", F.lit(None).cast("long"))
                .withColumn("attention_in_bin", F.lit(None).cast("double"))
                .withColumn("min_attention_in_bin", F.lit(None).cast("double"))
                .withColumn("max_attention_in_bin", F.lit(None).cast("double"))
                .withColumn("attention_rank", F.lit(None).cast("int"))
                .withColumn("rank_method", F.lit("not_computed_no_positive_primary_attention"))
                .withColumn("total_attention_in_frame", F.lit(0.0))
            )

        min_log = float(stats["min_log10_attention"] or 0.0)
        max_log = float(stats["max_log10_attention"] or min_log)
        n_bins = max(int(TRAFFIC_BLOCK_VALUE_BINS), int(n_blocks))
        bin_width = (max_log - min_log) / n_bins if max_log > min_log else 1.0
        bin_expr = F.least(
            F.floor((F.log10(F.col(attention_col)) - F.lit(min_log)) / F.lit(bin_width)).cast("int"),
            F.lit(n_bins - 1),
        )
        binned = base.withColumn("attention_value_bin", bin_expr).groupBy("attention_value_bin").agg(
            F.count("*").alias("n_channels_in_bin"),
            F.sum(attention_col).alias("attention_in_bin"),
            F.min(attention_col).alias("min_attention_in_bin"),
            F.max(attention_col).alias("max_attention_in_bin"),
        )
        pdf = binned.toPandas().sort_values("attention_value_bin", ascending=False).reset_index(drop=True)
        pdf["cum_attention_before_bin"] = pdf["attention_in_bin"].cumsum() - pdf["attention_in_bin"]
        pdf["cum_channels_before_bin"] = pdf["n_channels_in_bin"].cumsum() - pdf["n_channels_in_bin"]
        denom = float(total_attention) if total_attention else 1.0
        pdf["traffic_block"] = np.minimum(np.floor((pdf["cum_attention_before_bin"] / denom) * n_blocks).astype(int) + 1, n_blocks)
        pdf["approx_attention_rank_lower"] = pdf["cum_channels_before_bin"].astype(int) + 1
        pdf["approx_attention_rank_upper"] = (pdf["cum_channels_before_bin"] + pdf["n_channels_in_bin"]).astype(int)
        mapping = spark.createDataFrame(pdf[[
            "attention_value_bin", "traffic_block", "approx_attention_rank_lower", "approx_attention_rank_upper",
            "n_channels_in_bin", "attention_in_bin", "min_attention_in_bin", "max_attention_in_bin",
        ]])
        write_delta(mapping, "attention_value_bin_map")
        export_aggregate_csv(mapping, "attention_value_bin_map")

        status = spark.createDataFrame(pd.DataFrame([{
            "component": "attention_rank_blocks",
            "status": "computed_log_value_binned_approximation",
            "detail": "Traffic blocks and rank buckets are assigned through log-spaced value bins to avoid an unpartitioned global Spark orderBy window. Per-block actual view shares are exported separately.",
            "positive_channels": positive_channels,
            "total_attention": float(total_attention),
            "traffic_blocks": int(n_blocks),
            "traffic_block_value_bins": int(n_bins),
            "primary_attention_window_days": PRIMARY_ATTENTION_WINDOW,
            "min_primary_elapsed_days": MIN_PRIMARY_ELAPSED_DAYS,
            "allow_short_window_fallback": ALLOW_SHORT_WINDOW_FALLBACK,
            "execution_mode": EXECUTION_MODE,
        }]))
        write_delta(status, "attention_rank_status")
        export_aggregate_csv(status, "attention_rank_status")

        ranked = base.withColumn("attention_value_bin", bin_expr).join(F.broadcast(mapping), "attention_value_bin", "left").withColumn(
            "attention_rank", F.col("approx_attention_rank_lower")
        ).withColumn(
            "rank_method", F.lit("log_value_binned_approximation")
        ).withColumn(
            "total_attention_in_frame", F.lit(float(total_attention))
        )
        return ranked

    ranked_channels = add_attention_rank_blocks(channel_frame)
    write_internal_scratch_delta(ranked_channels, "ranked_channel_frame")

    traffic_block_summary = ranked_channels.groupBy("traffic_block").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
        F.sum("subscribers").alias("subscribers"),
        F.expr("percentile_approx(subscribers, 0.5, 1000)").alias("median_subscribers"),
        F.expr("percentile_approx(lifetime_views, 0.5, 1000)").alias("median_lifetime_views"),
        F.min("approx_attention_rank_lower").alias("approx_rank_min"),
        F.max("approx_attention_rank_upper").alias("approx_rank_max"),
    ).withColumn(
        "actual_view_share", F.col("weekly_views") / F.sum("weekly_views").over(Window.partitionBy())
    ).withColumn(
        "rank_method", F.lit("log_value_binned_approximation")
    ).orderBy("traffic_block")
    write_delta(traffic_block_summary, "traffic_block_summary")
    export_aggregate_csv(traffic_block_summary, "traffic_block_summary")
    display_if_enabled(traffic_block_summary, n=TRAFFIC_BLOCKS)
else:
    print('Skipped cell 19: execution_mode=manifest_only.')


## 4a. Compare Silver-Derived Attention to TOO Diagnostics

The TOO diagnostics layer is a useful QA surface, but this notebook does not treat it as authoritative without comparison. The next cell compares repeated-snapshot weekly attention with `views_past_year`, rank fields, and validation/sample-cut view shares where available.


In [ ]:
if RUN_COMPUTE:
    too_rank = read_table_optional(TABLES["diagnostics_rank_comparison"])
    if too_rank is not None and "channel_id" in too_rank.columns:
        if "run_id" in too_rank.columns:
            latest_too_rank_run = too_rank.agg(F.max("run_id").alias("run_id")).collect()[0]["run_id"]
            too_rank_use = too_rank.where(F.col("run_id") == latest_too_rank_run)
        else:
            latest_too_rank_run = None
            too_rank_use = too_rank
        too_cols = [
            F.col("channel_id").cast("string"),
            F.col("views_past_year").cast("double").alias("too_views_past_year") if "views_past_year" in too_rank_use.columns else F.lit(None).cast("double").alias("too_views_past_year"),
            F.col("rank_within_pastyear").cast("double").alias("too_rank_within_pastyear") if "rank_within_pastyear" in too_rank_use.columns else F.lit(None).cast("double").alias("too_rank_within_pastyear"),
            F.col("rank_overall_lifetime").cast("double").alias("too_rank_overall_lifetime") if "rank_overall_lifetime" in too_rank_use.columns else F.lit(None).cast("double").alias("too_rank_overall_lifetime"),
        ]
        too_compare = ranked_channels.select(
            "channel_id", "attention_rank", "weekly_views_primary", "lifetime_views", "subscribers", "subscriber_band"
        ).join(too_rank_use.select(*too_cols), "channel_id", "inner").withColumn(
            "annualized_weekly_views_primary", F.col("weekly_views_primary") * F.lit(52.0)
        )
        too_compare_summary = too_compare.agg(
            F.count("*").alias("n_overlap_channels"),
            F.corr("annualized_weekly_views_primary", "too_views_past_year").alias("corr_annualized_weekly_with_too_pastyear"),
            F.corr("attention_rank", "too_rank_within_pastyear").alias("corr_rank_with_too_pastyear_rank"),
            F.expr("percentile_approx(annualized_weekly_views_primary / nullif(too_views_past_year, 0), array(0.10,0.50,0.90), 1000)").alias("annualized_to_pastyear_ratio_p10_p50_p90"),
        ).withColumn("too_rank_run_id", F.lit(latest_too_rank_run))
        write_delta(too_compare_summary, "too_rank_attention_comparison")
        export_aggregate_csv(too_compare_summary, "too_rank_attention_comparison")

        too_compare_by_band = too_compare.groupBy("subscriber_band").agg(
            F.count("*").alias("n_overlap_channels"),
            F.corr("annualized_weekly_views_primary", "too_views_past_year").alias("corr_annualized_weekly_with_too_pastyear"),
            F.expr("percentile_approx(annualized_weekly_views_primary / nullif(too_views_past_year, 0), array(0.10,0.50,0.90), 1000)").alias("annualized_to_pastyear_ratio_p10_p50_p90"),
        ).withColumn("too_rank_run_id", F.lit(latest_too_rank_run))
        write_delta(too_compare_by_band, "too_rank_attention_comparison_by_band")
        export_aggregate_csv(too_compare_by_band, "too_rank_attention_comparison_by_band")
        display_if_enabled(too_compare_summary)
    else:
        status = spark.createDataFrame(pd.DataFrame([{"status": "too_rank_comparison_unavailable"}]))
        write_delta(status, "too_rank_attention_comparison")

    too_public = read_table_optional(TABLES["validation_too_public"])
    too_sample = read_table_optional(TABLES["validation_too_sample_cut"])
    validation_parts = []
    for label, df in [("yt_too_public", too_public), ("yt_too_sample_cut", too_sample)]:
        if df is not None and "canonical_id" in df.columns:
            select_exprs = [F.col("canonical_id").cast("string").alias("channel_id"), F.lit(label).alias("validation_source")]
            if "rank" in df.columns:
                select_exprs.append(F.col("rank").cast("double").alias("validation_rank"))
            else:
                select_exprs.append(F.lit(None).cast("double").alias("validation_rank"))
            if "view_count" in df.columns:
                select_exprs.append(F.col("view_count").cast("double").alias("validation_view_count"))
            else:
                select_exprs.append(F.lit(None).cast("double").alias("validation_view_count"))
            if "view_share" in df.columns:
                select_exprs.append(F.col("view_share").cast("double").alias("validation_view_share"))
            else:
                select_exprs.append(F.lit(None).cast("double").alias("validation_view_share"))
            validation_parts.append(df.select(*select_exprs))

    if validation_parts:
        validation_frame = validation_parts[0]
        for part in validation_parts[1:]:
            validation_frame = validation_frame.unionByName(part, allowMissingColumns=True)
        validation_comparison = ranked_channels.select("channel_id", "attention_rank", "weekly_views_primary", "subscribers").join(
            validation_frame, "channel_id", "inner"
        ).groupBy("validation_source").agg(
            F.count("*").alias("n_overlap_channels"),
            F.corr("attention_rank", "validation_rank").alias("corr_attention_rank_with_validation_rank"),
            F.corr("weekly_views_primary", "validation_view_count").alias("corr_weekly_views_with_validation_view_count"),
        )
        write_delta(validation_comparison, "too_validation_layer_comparison")
        export_aggregate_csv(validation_comparison, "too_validation_layer_comparison")
else:
    print('Skipped cell 21: execution_mode=manifest_only.')


## 5. Figure 1: Discovery Saturation and Collection Diagnostics

These diagnostics use collection-log and newly discovered channel tables. They are not a substitute for the final exhaustion proof, but they make the discovery curve, threshold crossing, and commercial-benchmark checks reproducible.


In [ ]:
if RUN_COMPUTE:
    def standardize_discovery_table(logical_name: str, source_label: str) -> Optional[DataFrame]:
        df = read_table_optional(TABLES[logical_name])
        if df is None:
            return None
        id_col = first_existing_column(df, ["canonical_id", "channel_id"], required=False, label="channel id")
        batch_col = first_existing_column(df, ["batch_number", "batch"], required=False, label="batch")
        if id_col is None:
            return None
        select_cols = [
            F.col(id_col).cast("string").alias("channel_id"),
            F.lit(source_label).alias("discovery_source"),
            F.col(batch_col).cast("int").alias("batch_number") if batch_col else F.lit(None).cast("int").alias("batch_number"),
        ]
        for source_col, out_col in [("subscriber_count", "subscribers"), ("view_count", "lifetime_views"), ("video_count", "lifetime_videos")]:
            if source_col in df.columns:
                select_cols.append(F.col(source_col).cast("double").alias(out_col))
            else:
                select_cols.append(F.lit(None).cast("double").alias(out_col))
        return df.select(*select_cols).where(F.col("channel_id").isNotNull())


    discovery_sources = [
        ("dev_new_channels", "default_new_channels"),
        ("dev_new_threshold_channels", "default_new_threshold_channels"),
        ("threshold_1k_new_channels", "threshold_1k_new_channels"),
        ("threshold_1k_new_threshold_channels", "threshold_1k_new_threshold_channels"),
        ("threshold_5k_new_channels", "threshold_5k_new_channels"),
        ("threshold_5k_new_threshold_channels", "threshold_5k_new_threshold_channels"),
    ]
    parts = [standardize_discovery_table(name, label) for name, label in discovery_sources]
    parts = [p for p in parts if p is not None]
    if parts:
        discovery_channels = parts[0]
        for part in parts[1:]:
            discovery_channels = discovery_channels.unionByName(part, allowMissingColumns=True)
    else:
        discovery_channels = empty_df_from_schema({
            "channel_id": "string",
            "discovery_source": "string",
            "batch_number": "integer",
            "subscribers": "double",
            "lifetime_views": "double",
            "lifetime_videos": "double",
        })

    discovery_channels = discovery_channels.withColumn("subscriber_band", subscriber_band_expr("subscribers")).withColumn("lifetime_view_band", view_band_expr("lifetime_views"))
    write_internal_scratch_delta(discovery_channels, "discovery_channels_standardized")

    discovery_by_batch = discovery_channels.groupBy("discovery_source", "batch_number", "subscriber_band", "lifetime_view_band").agg(
        F.countDistinct("channel_id").alias("new_channels"),
        F.sum(F.coalesce(F.col("lifetime_views"), F.lit(0.0))).alias("new_lifetime_views"),
        F.sum(F.coalesce(F.col("subscribers"), F.lit(0.0))).alias("new_subscribers"),
    ).orderBy("discovery_source", "batch_number")
    write_delta(discovery_by_batch, "discovery_by_batch")
    export_aggregate_csv(discovery_by_batch, "discovery_by_batch")

    batch_tables = []
    for logical_name, source_label in [
        ("dev_method_summary", "method_summary"),
        ("dev_trending_batches", "trending_batches"),
        ("dev_pub_subs_batches", "pub_subs_batches"),
        ("dev_pub_subs_full_pass_batches", "pub_subs_full_pass_batches"),
    ]:
        df = read_table_optional(TABLES[logical_name])
        if df is not None:
            batch_tables.append(df.withColumn("log_source", F.lit(source_label)))

    if batch_tables:
        collection_logs = batch_tables[0]
        for df in batch_tables[1:]:
            collection_logs = collection_logs.unionByName(df, allowMissingColumns=True)
        write_delta(collection_logs, "collection_batch_logs")
        export_aggregate_csv(collection_logs, "collection_batch_logs")

    # Benchmark overlap with SocialBlade/top-list style resources.
    sb = read_table_optional(TABLES["dev_socialblade_top50k"])
    top227 = read_table_optional(TABLES["dev_top_227k"])
    ranked_ids = ranked_channels.select("channel_id").dropDuplicates()
    benchmark_rows = []
    for label, df, id_candidates in [
        ("socialblade_top50k", sb, ["id.id", "canonical_id", "canonical_id_url"]),
        ("top_227k", top227, ["canonical_id", "user_id", "canonical_id_url"]),
    ]:
        if df is not None:
            id_col = first_existing_column(df, id_candidates, required=False, label=f"{label} id")
            if id_col:
                ids = df.select(F.col(id_col).cast("string").alias("channel_id")).where(F.col("channel_id").isNotNull()).dropDuplicates()
                n_ref = ids.count()
                n_overlap = ids.join(ranked_ids, "channel_id", "inner").count()
                benchmark_rows.append({"benchmark": label, "n_benchmark_channels": n_ref, "n_overlap_analysis_frame": n_overlap, "overlap_share": n_overlap / n_ref if n_ref else None})
    benchmark_df = spark.createDataFrame(pd.DataFrame(benchmark_rows)) if benchmark_rows else empty_df_from_schema({
        "benchmark": "string",
        "n_benchmark_channels": "long",
        "n_overlap_analysis_frame": "long",
        "overlap_share": "double",
    })
    write_delta(benchmark_df, "benchmark_overlap")
    export_aggregate_csv(benchmark_df, "benchmark_overlap")
    display_if_enabled(benchmark_df)
else:
    print('Skipped cell 23: execution_mode=manifest_only.')


## 6. Figure 2: Whole-Platform Composition and Treemap Aggregates

The treemap is generated from aggregate cells only. It labels large channels individually and pools smaller channels into language-category "Other" cells to avoid clutter and protect row-level exports.


In [ ]:
if RUN_COMPUTE:
    composition_block = ranked_channels.groupBy("traffic_block", "analysis_language", "analysis_category").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
        F.sum("subscribers").alias("subscribers"),
    ).withColumn(
        "traffic_block_label",
        F.concat(F.lit("Block "), F.col("traffic_block").cast("string")),
    )
    write_delta(composition_block, "composition_by_traffic_block")
    export_aggregate_csv(composition_block, "composition_by_traffic_block")

    language_category = ranked_channels.groupBy("analysis_language", "analysis_category").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
        F.sum("subscribers").alias("subscribers"),
    ).withColumn("view_share", F.col("weekly_views") / F.sum("weekly_views").over(Window.partitionBy()))
    write_delta(language_category, "language_category_composition")
    export_aggregate_csv(language_category, "language_category_composition")

    # Category-missingness bounds for hostile-reviewer robustness. Unknown/pending category mass is not silently ignored.
    unknown_category_values = ["", "unknown", "uncategorized", "none", "null", "pending"]
    category_audit_base = ranked_channels.withColumn(
        "category_is_unknown_or_pending",
        F.lower(F.coalesce(F.col("analysis_category"), F.lit("unknown"))).isin(unknown_category_values),
    )
    category_totals = category_audit_base.agg(
        F.sum("weekly_views_primary").alias("total_weekly_views"),
        F.sum(F.when(F.col("category_is_unknown_or_pending"), F.col("weekly_views_primary")).otherwise(F.lit(0.0))).alias("unknown_weekly_views"),
        F.count("*").alias("total_channels"),
        F.sum(F.when(F.col("category_is_unknown_or_pending"), 1).otherwise(0)).alias("unknown_channels"),
    ).collect()[0]
    _total_views = float(category_totals["total_weekly_views"] or 0.0)
    _unknown_views = float(category_totals["unknown_weekly_views"] or 0.0)
    _known_views = max(_total_views - _unknown_views, 0.0)
    _total_channels = int(category_totals["total_channels"] or 0)
    _unknown_channels = int(category_totals["unknown_channels"] or 0)
    _known_categories = category_audit_base.where(~F.col("category_is_unknown_or_pending")).groupBy("analysis_category").agg(
        F.count("*").alias("known_n_channels"),
        F.sum("weekly_views_primary").alias("known_weekly_views"),
    ).withColumn(
        "known_share_of_total_views_lower_bound", F.col("known_weekly_views") / F.lit(_total_views) if _total_views else F.lit(None).cast("double")
    ).withColumn(
        "mar_share_if_unknowns_like_known_categories", F.col("known_weekly_views") / F.lit(_known_views) if _known_views else F.lit(None).cast("double")
    ).withColumn(
        "adversarial_upper_bound_share", (F.col("known_weekly_views") + F.lit(_unknown_views)) / F.lit(_total_views) if _total_views else F.lit(None).cast("double")
    ).withColumn(
        "unknown_weekly_views", F.lit(_unknown_views)
    ).withColumn(
        "unknown_view_share", F.lit(_unknown_views / _total_views if _total_views else None).cast("double")
    ).withColumn(
        "unknown_channels", F.lit(_unknown_channels)
    ).withColumn(
        "unknown_channel_share", F.lit(_unknown_channels / _total_channels if _total_channels else None).cast("double")
    ).orderBy(F.col("known_weekly_views").desc())
    write_delta(_known_categories, "category_missingness_bounds")
    export_aggregate_csv(_known_categories, "category_missingness_bounds")

    # Quantify the "one platform or many" claim with pairwise Jensen-Shannon divergence among major language category distributions.
    def _jensen_shannon_divergence(p: np.ndarray, q: np.ndarray) -> float:
        p = np.asarray(p, dtype=float)
        q = np.asarray(q, dtype=float)
        if p.sum() <= 0 or q.sum() <= 0:
            return float("nan")
        p = p / p.sum()
        q = q / q.sum()
        m = 0.5 * (p + q)
        def _kl(a, b):
            mask = (a > 0) & (b > 0)
            return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))
        return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

    lc_pdf = language_category.select("analysis_language", "analysis_category", "weekly_views").toPandas()
    jsd_rows = []
    if not lc_pdf.empty:
        lang_totals = lc_pdf.groupby("analysis_language", dropna=False)["weekly_views"].sum().sort_values(ascending=False)
        total_views_lc = float(lang_totals.sum())
        major_langs = lang_totals[lang_totals / total_views_lc >= 0.005].head(12).index.tolist() if total_views_lc else []
        major_pdf = lc_pdf[lc_pdf["analysis_language"].isin(major_langs)].copy()
        pivot = major_pdf.pivot_table(index="analysis_language", columns="analysis_category", values="weekly_views", aggfunc="sum", fill_value=0.0)
        for i, lang_a in enumerate(pivot.index):
            for lang_b in list(pivot.index)[i + 1:]:
                jsd_rows.append({
                    "language_a": lang_a,
                    "language_b": lang_b,
                    "jensen_shannon_divergence_bits": _jensen_shannon_divergence(pivot.loc[lang_a].to_numpy(), pivot.loc[lang_b].to_numpy()),
                    "language_a_weekly_views": float(lang_totals.get(lang_a, 0.0)),
                    "language_b_weekly_views": float(lang_totals.get(lang_b, 0.0)),
                })
    if not jsd_rows:
        jsd_rows = [{"language_a": "not_available", "language_b": "not_available", "jensen_shannon_divergence_bits": float("nan"), "language_a_weekly_views": float("nan"), "language_b_weekly_views": float("nan")}]
    language_market_jsd = spark.createDataFrame(pd.DataFrame(jsd_rows))
    write_delta(language_market_jsd, "language_market_jsd")
    export_aggregate_csv(language_market_jsd, "language_market_jsd")

    total_attention = ranked_channels.agg(F.sum("weekly_views_primary").alias("weekly_views")).collect()[0]["weekly_views"] or 0.0
    channel_treemap_candidates = ranked_channels.select(
        "channel_id", "channel_name", "analysis_language", "analysis_category", "weekly_views_primary", "attention_rank"
    ).where(F.col("weekly_views_primary") > 0).withColumn("view_share", F.col("weekly_views_primary") / F.lit(float(total_attention)))

    channel_treemap_labeled = channel_treemap_candidates.where(
        (F.col("attention_rank") <= TOP_TREEMAP_CHANNELS) | (F.col("view_share") >= F.lit(TREEMAP_MIN_CHANNEL_SHARE))
    ).select(
        "analysis_language",
        "analysis_category",
        F.coalesce(F.col("channel_name"), F.col("channel_id")).alias("treemap_label"),
        F.col("weekly_views_primary").alias("weekly_views"),
        F.lit("channel").alias("cell_type"),
        F.count("*").over(Window.partitionBy("analysis_language", "analysis_category")).alias("_dummy"),
    ).drop("_dummy")

    channel_treemap_other = channel_treemap_candidates.where(
        ~((F.col("attention_rank") <= TOP_TREEMAP_CHANNELS) | (F.col("view_share") >= F.lit(TREEMAP_MIN_CHANNEL_SHARE)))
    ).groupBy("analysis_language", "analysis_category").agg(
        F.sum("weekly_views_primary").alias("weekly_views"),
        F.count("*").alias("n_channels_pooled"),
    ).where(F.col("weekly_views") > 0).select(
        "analysis_language",
        "analysis_category",
        F.concat(F.lit("Other channels (n="), F.col("n_channels_pooled").cast("string"), F.lit(")")).alias("treemap_label"),
        "weekly_views",
        F.lit("pooled_other").alias("cell_type"),
    )

    treemap_cells = channel_treemap_labeled.unionByName(channel_treemap_other, allowMissingColumns=True).withColumn(
        "view_share", F.col("weekly_views") / F.lit(float(total_attention))
    )
    write_delta(treemap_cells, "treemap_cells")
    export_aggregate_csv(treemap_cells, "treemap_cells")

    major_languages = language_category.groupBy("analysis_language").agg(F.sum("weekly_views").alias("weekly_views")).orderBy(F.col("weekly_views").desc()).limit(6)
    major_language_blocks = composition_block.join(major_languages.select("analysis_language"), "analysis_language", "inner")
    write_delta(major_language_blocks, "major_language_traffic_blocks")
    export_aggregate_csv(major_language_blocks, "major_language_traffic_blocks")
else:
    print('Skipped cell 25: execution_mode=manifest_only.')


## 7. Figure 3: Subscribers as an Imprecise Proxy for Attention

This section stores binned density and envelope tables rather than exporting raw scatterplot data. It computes interpretable summary statistics reviewers will ask for: rank correlations, log-log correlations, prediction bands, variance decomposition by major language/category, and the share of observed >=10k channels with inactive or unmeasured current attention by subscriber bin.


In [ ]:
if RUN_COMPUTE:
    proxy_base = ranked_channels.where(
        F.col("subscribers").isNotNull()
        & (F.col("subscribers") > 0)
        & (F.col("subscribers") >= F.lit(float(OBSERVED_ATTENTION_SUBSCRIBER_FLOOR)))
        & F.col("weekly_views_primary").isNotNull()
        & (F.col("weekly_views_primary") >= 0)
    ).withColumn("log10_subscribers", F.log10(F.col("subscribers") + F.lit(1.0))).withColumn(
        "log10_weekly_views", F.log10(F.col("weekly_views_primary") + F.lit(1.0))
    ).withColumn(
        "log10_views_per_subscriber", F.log10(F.col("views_per_subscriber_primary") + F.lit(1e-9))
    )

    proxy_summary = proxy_base.agg(
        F.count("*").alias("n_channels"),
        F.corr("log10_subscribers", "log10_weekly_views").alias("pearson_log_subs_log_views"),
        F.expr("corr(log10_subscribers, log10_views_per_subscriber)").alias("pearson_log_subs_log_views_per_sub"),
        F.expr("percentile_approx(weekly_views_primary, array(0.1, 0.5, 0.9), 1000)").alias("weekly_views_p10_p50_p90"),
        F.expr("percentile_approx(views_per_subscriber_primary, array(0.1, 0.5, 0.9), 1000)").alias("views_per_sub_p10_p50_p90"),
    )
    write_delta(proxy_summary, "subscriber_proxy_summary")
    export_aggregate_csv(proxy_summary, "subscriber_proxy_summary")

    proxy_density = proxy_base.withColumn(
        "log10_sub_bin", (F.floor(F.col("log10_subscribers") * F.lit(10)) / F.lit(10)).cast("double")
    ).withColumn(
        "log10_view_bin", (F.floor(F.col("log10_weekly_views") * F.lit(10)) / F.lit(10)).cast("double")
    ).groupBy("log10_sub_bin", "log10_view_bin").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
    )
    write_delta(proxy_density, "subscriber_proxy_density")
    export_aggregate_csv(proxy_density, "subscriber_proxy_density")

    proxy_envelope = proxy_base.withColumn(
        "log10_sub_bin", (F.floor(F.col("log10_subscribers") * F.lit(5)) / F.lit(5)).cast("double")
    ).groupBy("log10_sub_bin").agg(
        F.count("*").alias("n_channels"),
        F.expr("percentile_approx(weekly_views_primary, array(0.05,0.10,0.25,0.50,0.75,0.90,0.95), 1000)").alias("weekly_views_quantiles"),
        F.expr("percentile_approx(views_per_subscriber_primary, array(0.05,0.10,0.25,0.50,0.75,0.90,0.95), 1000)").alias("views_per_sub_quantiles"),
    ).where(F.col("n_channels") >= 20).orderBy("log10_sub_bin")
    write_delta(proxy_envelope, "subscriber_proxy_envelope")
    export_aggregate_csv(proxy_envelope, "subscriber_proxy_envelope")

    proxy_scope_all_observed = channel_frame.where(
        F.col("subscribers").isNotNull()
        & (F.col("subscribers") > 0)
        & (F.col("subscribers") >= F.lit(float(OBSERVED_ATTENTION_SUBSCRIBER_FLOOR)))
    ).withColumn(
        "log10_subscribers", F.log10(F.col("subscribers") + F.lit(1.0))
    ).withColumn(
        "weekly_views_primary_zero_filled", F.coalesce(F.col("weekly_views_primary"), F.lit(0.0))
    ).withColumn(
        "is_inactive_or_unmeasured",
        F.when(F.col("weekly_views_primary").isNull() | (F.col("weekly_views_primary") <= 0), F.lit(1)).otherwise(F.lit(0)),
    )
    proxy_inactive_share = proxy_scope_all_observed.withColumn(
        "log10_sub_bin", (F.floor(F.col("log10_subscribers") * F.lit(5)) / F.lit(5)).cast("double")
    ).groupBy("log10_sub_bin").agg(
        F.count("*").alias("n_channels"),
        F.sum("is_inactive_or_unmeasured").alias("inactive_or_unmeasured_channels"),
        F.avg("is_inactive_or_unmeasured").alias("inactive_or_unmeasured_share"),
        F.sum("weekly_views_primary_zero_filled").alias("weekly_views_zero_filled"),
    ).where(F.col("n_channels") >= 20).orderBy("log10_sub_bin")
    write_delta(proxy_inactive_share, "subscriber_proxy_inactive_share")
    export_aggregate_csv(proxy_inactive_share, "subscriber_proxy_inactive_share")

    proxy_by_language_category = proxy_base.groupBy("analysis_language", "analysis_category").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
        F.corr("log10_subscribers", "log10_weekly_views").alias("pearson_log_subs_log_views"),
        F.expr("percentile_approx(views_per_subscriber_primary, array(0.25,0.50,0.75), 1000)").alias("views_per_sub_iqr"),
    ).where(F.col("n_channels") >= 100)
    write_delta(proxy_by_language_category, "subscriber_proxy_by_language_category")
    export_aggregate_csv(proxy_by_language_category, "subscriber_proxy_by_language_category")
    display_if_enabled(proxy_summary)
else:
    print('Skipped cell 27: execution_mode=manifest_only.')


## 8. Figure 4: Subscriber Threshold Capture, Concentration, and Bounds

The main paper should lead with memorable threshold facts, but reviewers will also expect Lorenz/Gini diagnostics and sensitivity to subscriber cut points. This section writes both.


In [ ]:
if RUN_COMPUTE:
    observed_threshold_frame = channel_frame.where(
        F.col("subscribers").isNotNull()
        & (F.col("subscribers") >= F.lit(float(OBSERVED_ATTENTION_SUBSCRIBER_FLOOR)))
    ).withColumn(
        "weekly_views_for_threshold", F.coalesce(F.col("weekly_views_primary"), F.lit(0.0))
    ).cache()

    base_totals = observed_threshold_frame.agg(
        F.count("*").alias("total_channels"),
        F.sum("weekly_views_for_threshold").alias("total_weekly_views"),
        F.sum("subscribers").alias("total_subscribers"),
    ).collect()[0]

    total_channels = int(base_totals["total_channels"] or 0)
    total_weekly_views = float(base_totals["total_weekly_views"] or 0.0)
    total_subscribers = float(base_totals["total_subscribers"] or 0.0)
    threshold_denominator_label = f"all_observed_channels_at_or_above_{OBSERVED_ATTENTION_SUBSCRIBER_FLOOR}_subscribers_in_anchor_frame"

    threshold_rows = []
    for threshold in THRESHOLDS_SUBSCRIBERS:
        if threshold < OBSERVED_ATTENTION_SUBSCRIBER_FLOOR:
            threshold_rows.append({
                "threshold_subscribers": threshold,
                "channels": None,
                "weekly_views": None,
                "subscribers": None,
                "share_channels": None,
                "share_weekly_views": None,
                "share_subscribers": None,
                "inferential_tier": "design_or_bounded",
                "status": "below_observed_panel_floor_requires_threshold_strata_or_external_estimator",
                "denominator": "not_computable_from_observed_top_of_ocean_panel",
                "attention_handling": "requires_1k_strata_or_external_estimator",
            })
            continue
        row = observed_threshold_frame.where(F.col("subscribers") >= threshold).agg(
            F.count("*").alias("channels"),
            F.sum("weekly_views_for_threshold").alias("weekly_views"),
            F.sum("subscribers").alias("subscribers"),
        ).collect()[0]
        threshold_rows.append({
            "threshold_subscribers": threshold,
            "channels": int(row["channels"] or 0),
            "weekly_views": float(row["weekly_views"] or 0.0),
            "subscribers": float(row["subscribers"] or 0.0),
            "share_channels": (row["channels"] or 0) / total_channels if total_channels else None,
            "share_weekly_views": float(row["weekly_views"] or 0.0) / total_weekly_views if total_weekly_views else None,
            "share_subscribers": float(row["subscribers"] or 0.0) / total_subscribers if total_subscribers else None,
            "inferential_tier": "observed_head",
            "status": "observed_frame_share_not_platform_share",
            "denominator": threshold_denominator_label,
            "attention_handling": "weekly_views_primary_null_or_nonpositive_counted_as_zero_for_threshold_denominators",
        })
    threshold_capture = spark.createDataFrame(pd.DataFrame(threshold_rows))
    write_delta(threshold_capture, "threshold_capture")
    export_aggregate_csv(threshold_capture, "threshold_capture")

    rank_bin_pdf = load_output("attention_value_bin_map").select(
        "attention_value_bin",
        "n_channels_in_bin",
        "attention_in_bin",
        "approx_attention_rank_lower",
        "approx_attention_rank_upper",
    ).toPandas().sort_values("approx_attention_rank_lower")

    rank_rows = []
    if not rank_bin_pdf.empty:
        for col_name in ["n_channels_in_bin", "attention_in_bin", "approx_attention_rank_lower", "approx_attention_rank_upper"]:
            rank_bin_pdf[col_name] = pd.to_numeric(rank_bin_pdf[col_name], errors="coerce").fillna(0.0)
        positive_ranked_channels = int(rank_bin_pdf["n_channels_in_bin"].sum())
        positive_ranked_weekly_views = float(rank_bin_pdf["attention_in_bin"].sum())

        def cumulative_attention_at(top_n: int) -> Tuple[float, float, float]:
            if top_n <= 0 or positive_ranked_weekly_views <= 0 or positive_ranked_channels <= 0:
                return 0.0, 0.0, 0.0
            included_views = 0.0
            included_channels = 0.0
            capped_top_n = min(int(top_n), positive_ranked_channels)
            for row in rank_bin_pdf.itertuples(index=False):
                lower = int(row.approx_attention_rank_lower)
                upper = int(row.approx_attention_rank_upper)
                n_in_bin = int(row.n_channels_in_bin)
                views_in_bin = float(row.attention_in_bin)
                if capped_top_n < lower or n_in_bin <= 0:
                    break
                if capped_top_n >= upper:
                    take_channels = n_in_bin
                else:
                    take_channels = max(0, capped_top_n - lower + 1)
                fraction = min(max(take_channels / n_in_bin, 0.0), 1.0)
                included_channels += take_channels
                included_views += views_in_bin * fraction
                if capped_top_n < upper:
                    break
            return included_channels, included_views, included_views / positive_ranked_weekly_views

        rank_ranges = [
            (1, 100, "top_100"),
            (101, 1000, "top_1k"),
            (1001, 10000, "top_10k"),
            (10001, 100000, "top_100k"),
            (100001, positive_ranked_channels, "below_100k"),
        ]
        for lower, upper, label in rank_ranges:
            if upper < lower:
                bucket_channels = 0.0
                bucket_views = 0.0
                bucket_share = 0.0
                cumulative_share = 1.0 if positive_ranked_weekly_views > 0 and lower > positive_ranked_channels else 0.0
            else:
                lower_channels, lower_views, lower_share = cumulative_attention_at(lower - 1)
                upper_channels, upper_views, upper_share = cumulative_attention_at(upper)
                bucket_channels = max(upper_channels - lower_channels, 0.0)
                bucket_views = max(upper_views - lower_views, 0.0)
                bucket_share = max(upper_share - lower_share, 0.0)
                cumulative_share = upper_share
            rank_rows.append({
                "rank_bucket": label,
                "rank_lower": int(lower),
                "rank_upper": int(max(upper, lower - 1)),
                "n_channels": int(round(bucket_channels)),
                "weekly_views": float(bucket_views),
                "view_share": float(bucket_share),
                "cumulative_view_share": float(cumulative_share),
                "denominator": "positive_attention_ranked_channels",
                "rank_method": "log_value_bin_interpolated",
                "status": "interpolated_from_attention_value_bin_map_not_exact_channel_rank",
            })
    rank_capture = spark.createDataFrame(pd.DataFrame(rank_rows)) if rank_rows else empty_df_from_schema({
        "rank_bucket": "string",
        "rank_lower": "integer",
        "rank_upper": "integer",
        "n_channels": "integer",
        "weekly_views": "double",
        "view_share": "double",
        "cumulative_view_share": "double",
        "denominator": "string",
        "rank_method": "string",
        "status": "string",
    })
    write_delta(rank_capture, "rank_capture")
    export_aggregate_csv(rank_capture, "rank_capture")

    # Scalable Lorenz approximation from log-spaced value bins, avoiding ntile over the full channel frame.
    positive_attention = ranked_channels.where(F.col("weekly_views_primary") > 0).withColumn("log10_weekly_views", F.log10("weekly_views_primary"))
    lorenz_bounds = positive_attention.agg(
        F.count("*").alias("positive_channels"),
        F.sum("weekly_views_primary").alias("positive_weekly_views"),
        F.min("log10_weekly_views").alias("min_log10_views"),
        F.max("log10_weekly_views").alias("max_log10_views"),
    ).collect()[0]
    positive_channels = int(lorenz_bounds["positive_channels"] or 0)
    positive_weekly_views = float(lorenz_bounds["positive_weekly_views"] or 0.0)
    if positive_channels and positive_weekly_views > 0:
        min_log10_views = float(lorenz_bounds["min_log10_views"] or 0.0)
        max_log10_views = float(lorenz_bounds["max_log10_views"] or min_log10_views)
        n_lorenz_bins = 1000
        width = (max_log10_views - min_log10_views) / n_lorenz_bins if max_log10_views > min_log10_views else 1.0
        lorenz_bins = positive_attention.withColumn(
            "lorenz_bin",
            F.least(
                F.floor((F.col("log10_weekly_views") - F.lit(min_log10_views)) / F.lit(width)).cast("int"),
                F.lit(n_lorenz_bins - 1),
            ),
        ).groupBy("lorenz_bin").agg(
            F.count("*").alias("n_channels"),
            F.sum("weekly_views_primary").alias("weekly_views"),
            F.avg("weekly_views_primary").alias("mean_weekly_views"),
        ).withColumn(
            "bin_midpoint_weekly_views", F.pow(F.lit(10.0), F.lit(min_log10_views) + (F.col("lorenz_bin") + F.lit(0.5)) * F.lit(width))
        ).orderBy("lorenz_bin")
        write_delta(lorenz_bins, "lorenz_value_bins")
        export_aggregate_csv(lorenz_bins, "lorenz_value_bins")
        w_lorenz = Window.orderBy("lorenz_bin").rowsBetween(Window.unboundedPreceding, Window.currentRow)
        lorenz_curve = lorenz_bins.withColumn("cum_channels", F.sum("n_channels").over(w_lorenz)).withColumn(
            "cum_views", F.sum("weekly_views").over(w_lorenz)
        ).withColumn(
            "cum_channel_share", F.col("cum_channels") / F.lit(positive_channels)
        ).withColumn(
            "cum_view_share", F.col("cum_views") / F.lit(positive_weekly_views)
        ).withColumn("denominator", F.lit("positive_attention_ranked_channels"))
    else:
        lorenz_curve = empty_df_from_schema({
            "lorenz_bin": "integer",
            "n_channels": "long",
            "weekly_views": "double",
            "mean_weekly_views": "double",
            "bin_midpoint_weekly_views": "double",
            "cum_channels": "long",
            "cum_views": "double",
            "cum_channel_share": "double",
            "cum_view_share": "double",
            "denominator": "string",
        })
    write_delta(lorenz_curve, "lorenz_curve")
    export_aggregate_csv(lorenz_curve, "lorenz_curve")

    # Gini from value-binned Lorenz curve. This is an approximation for diagnostics; exact channel-level Gini can be added if reviewers request it.
    pdf_lorenz = lorenz_curve.select("cum_channel_share", "cum_view_share").toPandas().sort_values("cum_channel_share")
    if len(pdf_lorenz):
        x = np.r_[0.0, pdf_lorenz["cum_channel_share"].to_numpy(), 1.0]
        y = np.r_[0.0, pdf_lorenz["cum_view_share"].to_numpy(), 1.0]
        gini_approx = 1.0 - 2.0 * np.trapz(y, x)
    else:
        gini_approx = float("nan")
    gini_df = spark.createDataFrame(pd.DataFrame([{"gini_weekly_views_lorenz_value_binned": float(gini_approx), "denominator": "positive_attention_ranked_channels"}]))
    write_delta(gini_df, "concentration_gini")
    export_aggregate_csv(gini_df, "concentration_gini")
    display_if_enabled(threshold_capture)
else:
    print('Skipped cell 29: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    # Optional bounded-tail / design-based hook. This writes an explicit status table if the exact sample design table is not yet specified.
    design_status_rows = []
    if SAMPLE_DESIGN_TABLE_FULL_NAME and SAMPLE_WEIGHT_COL and SAMPLE_STRATUM_COL:
        design = spark.table(SAMPLE_DESIGN_TABLE_FULL_NAME)
        required = {SAMPLE_UNIT_COL, SAMPLE_WEIGHT_COL, SAMPLE_STRATUM_COL}
        missing = [c for c in required if c not in design.columns]
        if missing:
            design_status_rows.append({"component": "sample_design", "status": "missing_columns", "detail": ",".join(missing)})
        else:
            sample_frame = ranked_channels.join(
                design.select(
                    F.col(SAMPLE_UNIT_COL).cast("string").alias("channel_id"),
                    F.col(SAMPLE_WEIGHT_COL).cast("double").alias("sample_weight"),
                    F.col(SAMPLE_STRATUM_COL).cast("string").alias("sample_stratum"),
                ),
                "channel_id",
                "inner",
            )
            design_estimates = sample_frame.groupBy("sample_stratum").agg(
                F.count("*").alias("n_sample_channels"),
                F.sum(F.col("sample_weight") * F.col("weekly_views_primary")).alias("weighted_weekly_views"),
                F.sum(F.col("sample_weight")).alias("estimated_channels"),
            )
            write_delta(design_estimates, "design_based_estimates_by_stratum")
            export_aggregate_csv(design_estimates, "design_based_estimates_by_stratum")
            design_status_rows.append({"component": "sample_design", "status": "computed", "detail": SAMPLE_DESIGN_TABLE_FULL_NAME})
    else:
        design_status_rows.append({
            "component": "sample_design",
            "status": "not_configured",
            "detail": "Set sample_design_table_full_name, sample_weight_col, and sample_stratum_col for design-based weighted lower-rank estimates.",
        })

    bounds_rows = [{
        "bound_component": "residual_tail_subscriber_floor",
        "lower_bound": 1000.0,
        "upper_bound": None,
        "status": "external_estimator_not_in_repo",
    }]
    write_delta(spark.createDataFrame(pd.DataFrame(design_status_rows)), "sample_design_status")
    write_delta(spark.createDataFrame(pd.DataFrame(bounds_rows)), "residual_tail_bounds")
else:
    print('Skipped cell 30: execution_mode=manifest_only.')


## 9. Figure 5: Channel Age / Start-Date Structure

True channel founding dates are not exposed in the current silver channel table. This notebook uses the best available configured source in the following order: SocialBlade-style channel `created_at` for overlapping channels, then earliest observed video publication date as a lower-bound proxy. The output retains a `channel_start_source` flag so the Methods section can be precise.


In [ ]:
if RUN_COMPUTE:
    def build_socialblade_created_at() -> DataFrame:
        sb = read_table_optional(TABLES["dev_socialblade_top50k"])
        if sb is None:
            return empty_df_from_schema({"channel_id": "string", "channel_created_at_socialblade": "timestamp"})
        id_col = first_existing_column(sb, ["id.id", "canonical_id", "canonical_id_url"], required=False, label="socialblade id")
        created_col = first_existing_column(sb, ["general.created_at", "created_at"], required=False, label="socialblade created_at")
        if id_col is None or created_col is None:
            return empty_df_from_schema({"channel_id": "string", "channel_created_at_socialblade": "timestamp"})
        return sb.select(
            F.col(id_col).cast("string").alias("channel_id"),
            F.col(created_col).cast("timestamp").alias("channel_created_at_socialblade"),
        ).where(F.col("channel_id").isNotNull()).dropDuplicates(["channel_id"])


    def build_earliest_video_date(channel_ids: DataFrame) -> DataFrame:
        videos = spark.table(TABLES["videos"]).select(
            F.col("channel_id").cast("string"),
            F.col("published_at").cast("timestamp").alias("published_at"),
            F.to_date("published_date").alias("published_date"),
        ).where(F.col("channel_id").isNotNull())
        if EXECUTION_MODE == "smoke":
            videos = videos.join(F.broadcast(channel_ids.select("channel_id").limit(SMOKE_CHANNEL_LIMIT)), "channel_id", "inner")
        return videos.groupBy("channel_id").agg(F.min(F.coalesce(F.col("published_at"), F.col("published_date").cast("timestamp"))).alias("earliest_observed_video_at"))

    age_sources = build_socialblade_created_at().join(build_earliest_video_date(ranked_channels.select("channel_id")), "channel_id", "full")
    age_frame = ranked_channels.join(age_sources, "channel_id", "left").withColumn(
        "channel_start_at",
        F.coalesce(F.col("channel_created_at_socialblade"), F.col("earliest_observed_video_at")),
    ).withColumn(
        "channel_start_source",
        F.when(F.col("channel_created_at_socialblade").isNotNull(), F.lit("socialblade_channel_created_at"))
        .when(F.col("earliest_observed_video_at").isNotNull(), F.lit("earliest_observed_video_proxy"))
        .otherwise(F.lit("missing")),
    ).withColumn("channel_start_year", F.year("channel_start_at"))

    age_summary = age_frame.groupBy("traffic_block", "channel_start_source").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
        F.expr("percentile_approx(channel_start_year, array(0.10,0.25,0.50,0.75,0.90), 1000)").alias("start_year_quantiles"),
        F.avg("channel_start_year").alias("mean_start_year"),
    ).orderBy("traffic_block", "channel_start_source")
    write_delta(age_summary, "age_summary_by_traffic_block")
    export_aggregate_csv(age_summary, "age_summary_by_traffic_block")

    age_histogram = age_frame.where(F.col("channel_start_year").isNotNull()).groupBy("traffic_block", "channel_start_year", "channel_start_source").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
    )
    write_delta(age_histogram, "age_histogram_by_traffic_block")
    export_aggregate_csv(age_histogram, "age_histogram_by_traffic_block")
    display_if_enabled(age_summary, n=100)


    # JNS alternative for Figure 5: language rank, attention, and speaker-population scale.
    def language_population_key_expr(label_col: str = "analysis_language", iso_col: str = "consensus_language_iso639_3") -> F.Column:
        iso_value = F.lower(F.substring(F.col(iso_col).cast("string"), 1, 3)) if iso_col in ranked_channels.columns else F.lit(None).cast("string")
        label_prefix = F.regexp_extract(F.lower(F.col(label_col).cast("string")), r"^([a-z]{3})(?:_|$)", 1)
        label_slug = F.lower(F.regexp_replace(F.coalesce(F.col(label_col).cast("string"), F.lit("unknown")), r"[^A-Za-z0-9]+", "_"))
        return F.coalesce(
            F.when(F.length(iso_value) == 3, iso_value),
            F.when(F.length(label_prefix) == 3, label_prefix),
            label_slug,
        )


    def build_speaker_population_lookup() -> DataFrame:
        if SPEAKER_POPULATION_TABLE_FULL_NAME and table_exists(SPEAKER_POPULATION_TABLE_FULL_NAME):
            pop = spark.table(SPEAKER_POPULATION_TABLE_FULL_NAME)
            lang_col = first_existing_column(pop, ["iso639_3", "language_code", "analysis_language", "language", "language_name"], required=True, label="speaker population language")
            pop_col = first_existing_column(pop, ["speaker_population_millions", "population_millions", "speakers_millions", "speakers_m"], required=True, label="speaker population")
            raw_key = F.lower(F.regexp_replace(F.coalesce(F.col(lang_col).cast("string"), F.lit("unknown")), r"[^A-Za-z0-9]+", "_"))
            return pop.select(
                F.coalesce(
                    F.when(F.length(F.substring(raw_key, 1, 3)) == 3, F.substring(raw_key, 1, 3)),
                    raw_key,
                ).alias("speaker_population_key"),
                F.col(pop_col).cast("double").alias("speaker_population_millions"),
                F.lit(SPEAKER_POPULATION_TABLE_FULL_NAME).alias("population_source_note"),
            ).dropDuplicates(["speaker_population_key"])
        # Approximate first-language plus second-language speaker scale, for diagnostic plotting only.
        rows = [
            ("eng", 1500.0), ("cmn", 1100.0), ("zho", 1100.0), ("hin", 610.0),
            ("spa", 560.0), ("ara", 360.0), ("fra", 310.0), ("fre", 310.0),
            ("ben", 280.0), ("por", 260.0), ("rus", 255.0), ("ind", 200.0),
            ("urd", 230.0), ("deu", 135.0), ("ger", 135.0), ("jpn", 125.0),
            ("vie", 100.0), ("kor", 82.0), ("tur", 88.0), ("tam", 85.0),
            ("tel", 96.0), ("tha", 61.0), ("ita", 65.0), ("mar", 100.0),
            ("pcm", 120.0),
        ]
        return spark.createDataFrame(pd.DataFrame(rows, columns=["speaker_population_key", "speaker_population_millions"])).withColumn(
            "population_source_note", F.lit("approximate_order_of_magnitude_default; replace with approved table before publication")
        )

    language_attention_group_cols = ["analysis_language"]
    language_aggs = [
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
        F.sum("subscribers").alias("subscribers"),
    ]
    if "consensus_language_iso639_3" in ranked_channels.columns:
        language_aggs.insert(0, F.first("consensus_language_iso639_3", ignorenulls=True).alias("consensus_language_iso639_3"))
    language_attention = ranked_channels.groupBy(*language_attention_group_cols).agg(*language_aggs)
    if "consensus_language_iso639_3" not in language_attention.columns:
        language_attention = language_attention.withColumn("consensus_language_iso639_3", F.lit(None).cast("string"))
    language_attention = language_attention.withColumn(
        "view_share", F.col("weekly_views") / F.sum("weekly_views").over(Window.partitionBy())
    ).withColumn(
        "language_rank_by_attention", F.row_number().over(Window.orderBy(F.col("weekly_views").desc_nulls_last()))
    ).withColumn("speaker_population_key", language_population_key_expr("analysis_language", "consensus_language_iso639_3"))

    language_rank_population = language_attention.join(build_speaker_population_lookup(), "speaker_population_key", "left").withColumn(
        "weekly_views_per_million_speakers", F.col("weekly_views") / F.col("speaker_population_millions")
    ).orderBy("language_rank_by_attention")
    write_delta(language_rank_population, "language_rank_engagement_population")
    export_aggregate_csv(language_rank_population, "language_rank_engagement_population")
else:
    print('Skipped cell 32: execution_mode=manifest_only.')


## 10. Figure 6: Production and Recent-Upload View Stock by Format

Shorts and long-form views are different regimes. The default Shorts rule here is duration <= 180 seconds, matching current three-minute Shorts eligibility. The production panel uses uploads in the configured window. The format-share panel is explicitly the stock of cumulative views on videos published in the recent lookback window, not a within-window attention delta; video-level delta attention should replace it if a complete video metric panel becomes available. The robustness section repeats the format analysis with a 60-second cutoff.


In [ ]:
if RUN_COMPUTE:
    def duration_seconds_expr(video_length_col: str = "video_length") -> F.Column:
        raw = F.col(video_length_col).cast("string")
        iso_hours = F.regexp_extract(raw, r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+(?:\.\d+)?)S)?", 1).cast("double")
        iso_minutes = F.regexp_extract(raw, r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+(?:\.\d+)?)S)?", 2).cast("double")
        iso_seconds = F.regexp_extract(raw, r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+(?:\.\d+)?)S)?", 3).cast("double")
        iso_total = F.coalesce(iso_hours, F.lit(0.0)) * 3600 + F.coalesce(iso_minutes, F.lit(0.0)) * 60 + F.coalesce(iso_seconds, F.lit(0.0))
        colon_parts = F.split(raw, ":")
        colon_n = F.size(colon_parts)
        colon_total = (
            F.when(colon_n == 3, colon_parts.getItem(0).cast("double") * 3600 + colon_parts.getItem(1).cast("double") * 60 + colon_parts.getItem(2).cast("double"))
            .when(colon_n == 2, colon_parts.getItem(0).cast("double") * 60 + colon_parts.getItem(1).cast("double"))
            .when(colon_n == 1, colon_parts.getItem(0).cast("double"))
        )
        return F.when(raw.rlike(r"^PT"), iso_total).otherwise(colon_total)


    def _scoped_ranked_channel_ids() -> DataFrame:
        ids = ranked_channels.select("channel_id").dropDuplicates()
        if EXECUTION_MODE == "smoke":
            return F.broadcast(ids.limit(SMOKE_CHANNEL_LIMIT))
        return ids


    def empty_video_base_frame() -> DataFrame:
        return empty_df_from_schema({
            "channel_id": "string",
            "video_id": "string",
            "video_title": "string",
            "post_type": "string",
            "video_length": "string",
            "video_category": "string",
            "published_at": "timestamp",
            "published_date": "date",
            "duration_seconds": "double",
            "video_views_latest": "double",
        })


    def build_video_base_frame() -> DataFrame:
        """Build the recent-upload video frame once; Shorts cutoffs are reclassified from this frame."""
        videos_raw = spark.table(TABLES["videos"])
        category_col = first_existing_column(videos_raw, CATEGORY_CANDIDATE_COLS, required=False, label="video category")
        videos = videos_raw.select(
            F.col("channel_id").cast("string"),
            F.col("video_id").cast("string"),
            F.col("video_title").cast("string"),
            F.col("post_type").cast("string"),
            F.col("video_length").cast("string"),
            F.col(category_col).cast("string").alias("video_category") if category_col else F.lit(None).cast("string").alias("video_category"),
            F.col("published_at").cast("timestamp"),
            F.to_date("published_date").alias("published_date"),
        ).where(F.col("channel_id").isNotNull() & F.col("video_id").isNotNull())
        format_anchor_date = SELECTED_ATTENTION_ANCHOR_DATE or TARGET_CAPTURE_DATE
        if format_anchor_date:
            videos = videos.where(F.col("published_date") <= F.to_date(F.lit(format_anchor_date)))
        if FORMAT_LOOKBACK_DAYS > 0:
            anchor_date = F.to_date(F.lit(format_anchor_date)) if format_anchor_date else F.current_date()
            videos = videos.where(F.col("published_date") >= F.date_sub(anchor_date, FORMAT_LOOKBACK_DAYS))
        videos = videos.join(_scoped_ranked_channel_ids(), "channel_id", "inner")

        videos = videos.withColumn("duration_seconds", duration_seconds_expr("video_length"))

        vm = spark.table(TABLES["video_metrics"]).select(
            F.col("channel_id").cast("string"),
            F.col("video_id").cast("string"),
            F.col("view_count").cast("double").alias("video_views_latest"),
            F.to_date("capture_date").alias("capture_date"),
        ).where(F.col("channel_id").isNotNull() & F.col("video_id").isNotNull())
        if format_anchor_date:
            vm = vm.where(F.col("capture_date") <= F.to_date(F.lit(format_anchor_date)))
        vm = vm.join(_scoped_ranked_channel_ids(), "channel_id", "inner")
        w = Window.partitionBy("channel_id", "video_id").orderBy(F.col("capture_date").desc_nulls_last())
        vm_latest = vm.withColumn("rn", F.row_number().over(w)).where(F.col("rn") == 1).drop("rn", "capture_date")
        return videos.dropDuplicates(["channel_id", "video_id"]).join(vm_latest, ["channel_id", "video_id"], "left")


    def classify_video_format(base_frame: DataFrame, shorts_cutoff_seconds: int = SHORTS_MAX_SECONDS) -> DataFrame:
        base = base_frame.drop("format") if "format" in base_frame.columns else base_frame
        return base.withColumn(
            "format",
            F.when(F.col("duration_seconds").isNotNull() & (F.col("duration_seconds") <= shorts_cutoff_seconds), F.lit("short_form"))
            .when(F.lower(F.coalesce(F.col("post_type"), F.lit(""))).contains("short"), F.lit("short_form"))
            .otherwise(F.lit("long_form_or_unknown")),
        )


    def build_video_format_frame(shorts_cutoff_seconds: int = SHORTS_MAX_SECONDS, base_frame: Optional[DataFrame] = None) -> DataFrame:
        base = base_frame if base_frame is not None else build_video_base_frame()
        return classify_video_format(base, shorts_cutoff_seconds)


    if ranked_channels.limit(1).count() == 0:
        warn_manifest("Video production/format frame skipped because no positive-attention ranked channels are available.")
        video_base_frame = empty_video_base_frame()
    else:
        video_base_frame = build_video_base_frame().cache()
    video_format_frame = build_video_format_frame(SHORTS_MAX_SECONDS, video_base_frame)
    write_internal_scratch_delta(video_format_frame, "video_format_frame")

    production_by_channel = video_format_frame.groupBy("channel_id", "format").agg(
        F.countDistinct("video_id").alias("uploads_in_format_window"),
        F.sum(F.coalesce(F.col("video_views_latest"), F.lit(0.0))).alias("video_views_latest_in_format_window"),
        F.expr("percentile_approx(duration_seconds, 0.5, 1000)").alias("median_duration_seconds"),
    )
    production_joined = ranked_channels.select(
        "channel_id", "traffic_block", "attention_rank", "weekly_views_primary", "analysis_language", "analysis_category", "subscribers"
    ).join(production_by_channel, "channel_id", "left")

    production_summary = production_joined.groupBy("traffic_block", "format").agg(
        F.countDistinct("channel_id").alias("n_channels"),
        F.sum("uploads_in_format_window").alias("uploads"),
        F.sum("video_views_latest_in_format_window").alias("video_views_latest"),
        F.sum("weekly_views_primary").alias("channel_weekly_views"),
        F.expr("percentile_approx(uploads_in_format_window, array(0.25,0.50,0.75), 1000)").alias("uploads_iqr"),
    ).orderBy("traffic_block", "format")
    write_delta(production_summary, "production_summary_by_traffic_block_format")
    export_aggregate_csv(production_summary, "production_summary_by_traffic_block_format")

    format_share = video_format_frame.groupBy("format").agg(
        F.countDistinct("video_id").alias("n_videos"),
        F.sum(F.coalesce(F.col("video_views_latest"), F.lit(0.0))).alias("video_views_latest"),
    ).withColumn(
        "cumulative_recent_upload_view_share",
        F.col("video_views_latest") / F.sum("video_views_latest").over(Window.partitionBy()),
    ).withColumn(
        "video_view_share", F.col("cumulative_recent_upload_view_share")
    ).withColumn(
        "view_measure", F.lit("cumulative_views_on_uploads_published_within_lookback_not_within_window_attention")
    ).withColumn(
        "lookback_days", F.lit(int(FORMAT_LOOKBACK_DAYS))
    )
    write_delta(format_share, "format_share")
    export_aggregate_csv(format_share, "format_share")
    display_if_enabled(format_share)
else:
    print('Skipped cell 34: execution_mode=manifest_only.')


## 11. Robustness Checks and Reviewer Diagnostics

The checks below are deliberately redundant. A hostile reviewer will ask whether headline claims survive plausible changes in attention windows, label definitions, suspicious-channel exclusions, category source choices, and Shorts thresholds.


In [ ]:
if RUN_COMPUTE:
    # Attention-window robustness for threshold capture and subscriber correlation.
    window_robust_rows = []
    for window_days in ATTENTION_WINDOWS:
        attention_col = f"weekly_views_{window_days}d"
        if attention_col not in channel_frame.columns:
            continue
        df = channel_frame.where(F.col(attention_col).isNotNull() & (F.col(attention_col) > 0))
        totals = df.agg(F.count("*").alias("channels"), F.sum(attention_col).alias("views")).collect()[0]
        total_v = float(totals["views"] or 0.0)
        corr_row = df.where(F.col("subscribers") > 0).withColumn("log_sub", F.log10(F.col("subscribers") + 1)).withColumn(
            "log_views", F.log10(F.col(attention_col) + 1)
        ).agg(F.corr("log_sub", "log_views").alias("corr")).collect()[0]
        for threshold in THRESHOLDS_SUBSCRIBERS:
            if threshold < OBSERVED_ATTENTION_SUBSCRIBER_FLOOR:
                window_robust_rows.append({
                    "window_days": window_days,
                    "threshold_subscribers": threshold,
                    "n_channels": None,
                    "view_share": None,
                    "log_subscriber_attention_correlation": float(corr_row["corr"]) if corr_row["corr"] is not None else None,
                    "status": "below_observed_panel_floor_requires_threshold_strata_or_external_estimator",
                })
                continue
            cap = df.where(F.col("subscribers") >= threshold).agg(F.count("*").alias("channels"), F.sum(attention_col).alias("views")).collect()[0]
            window_robust_rows.append({
                "window_days": window_days,
                "threshold_subscribers": threshold,
                "n_channels": int(cap["channels"] or 0),
                "view_share": float(cap["views"] or 0.0) / total_v if total_v else None,
                "log_subscriber_attention_correlation": float(corr_row["corr"]) if corr_row["corr"] is not None else None,
                "status": "observed_frame_share_not_platform_share",
            })
    window_robustness = spark.createDataFrame(pd.DataFrame(window_robust_rows))
    write_delta(window_robustness, "robustness_attention_windows")
    export_aggregate_csv(window_robustness, "robustness_attention_windows")

    # Suspicious-channel diagnostic sensitivity. Project decision: flags are diagnostic-only; they are not exclusions in main estimates.
    suspicion = read_table_optional(TABLES["diagnostics_suspicion_flags"])
    if suspicion is not None and "channel_id" in suspicion.columns:
        latest_run_col = "run_id" if "run_id" in suspicion.columns else None
        if latest_run_col:
            latest_diag_run = suspicion.agg(F.max("run_id").alias("run_id")).collect()[0]["run_id"]
            suspicion_use = suspicion.where(F.col("run_id") == latest_diag_run)
        else:
            latest_diag_run = None
            suspicion_use = suspicion
        flag_col = "n_flags_fired" if "n_flags_fired" in suspicion_use.columns else None
        diag = suspicion_use.select(
            F.col("channel_id").cast("string"),
            F.col(flag_col).cast("int").alias("n_flags_fired") if flag_col else F.lit(None).cast("int").alias("n_flags_fired"),
        ).dropDuplicates(["channel_id"])
        with_flags = ranked_channels.join(diag, "channel_id", "left").withColumn("exclude_suspicious", F.coalesce(F.col("n_flags_fired"), F.lit(0)) > 0)
        sensitivity_rows = []
        for label, condition in [("all_channels", F.lit(True)), ("diagnostic_counterfactual_exclude_any_suspicion_flag", ~F.col("exclude_suspicious"))]:
            sub = with_flags.where(condition)
            totals = sub.agg(F.count("*").alias("channels"), F.sum("weekly_views_primary").alias("views")).collect()[0]
            sensitivity_rows.append({
                "sample": label,
                "diagnostic_run_id": latest_diag_run,
                "n_channels": int(totals["channels"] or 0),
                "weekly_views": float(totals["views"] or 0.0),
            })
        suspicion_sensitivity = spark.createDataFrame(pd.DataFrame(sensitivity_rows))
    else:
        suspicion_sensitivity = spark.createDataFrame(pd.DataFrame([{"sample": "not_available", "diagnostic_run_id": None, "n_channels": None, "weekly_views": None}]))
    write_delta(suspicion_sensitivity, "robustness_suspicion_exclusion")
    export_aggregate_csv(suspicion_sensitivity, "robustness_suspicion_exclusion")

    # Language source sensitivity: compare source language, detected language, and LID consensus where available.
    language_source_summary = channel_frame.select(
        "channel_id", "weekly_views_primary", "language_code", "detected_language", "analysis_language", "consensus_status"
    ).groupBy("language_code", "detected_language", "analysis_language", "consensus_status").agg(
        F.count("*").alias("n_channels"),
        F.sum("weekly_views_primary").alias("weekly_views"),
    )
    write_delta(language_source_summary, "robustness_language_source_summary")
    export_aggregate_csv(language_source_summary, "robustness_language_source_summary")
else:
    print('Skipped cell 36: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    # Shorts definition sensitivity. Reclassify the already-built recent-upload frame; do not rescan videos.
    format_sensitivity_rows = []
    if "video_base_frame" not in globals():
        video_base_frame = build_video_base_frame().cache()
    for cutoff in [60, 180]:
        vf = build_video_format_frame(cutoff, video_base_frame)
        fs = vf.groupBy("format").agg(
            F.countDistinct("video_id").alias("n_videos"),
            F.sum(F.coalesce(F.col("video_views_latest"), F.lit(0.0))).alias("video_views_latest"),
        ).withColumn("shorts_cutoff_seconds", F.lit(cutoff))
        format_sensitivity_rows.append(fs)
    format_sensitivity = format_sensitivity_rows[0]
    for part in format_sensitivity_rows[1:]:
        format_sensitivity = format_sensitivity.unionByName(part, allowMissingColumns=True)
    format_sensitivity = format_sensitivity.withColumn(
        "view_share_within_cutoff",
        F.col("video_views_latest") / F.sum("video_views_latest").over(Window.partitionBy("shorts_cutoff_seconds")),
    )
    write_delta(format_sensitivity, "robustness_shorts_cutoff")
    export_aggregate_csv(format_sensitivity, "robustness_shorts_cutoff")

    # Missingness and negative-delta audit.
    neg_cols = [c for c in channel_frame.columns if c.startswith("negative_delta_")]
    missingness_exprs = [
        F.count("*").alias("n_channels"),
        F.sum(F.when(F.col("weekly_views_primary").isNull(), 1).otherwise(0)).alias("missing_primary_attention"),
        F.sum(F.when(F.col("subscribers").isNull(), 1).otherwise(0)).alias("missing_subscribers"),
        F.sum(F.when(F.col("analysis_language") == "unknown", 1).otherwise(0)).alias("unknown_language"),
        F.sum(F.when(F.col("analysis_category") == "unknown", 1).otherwise(0)).alias("unknown_category"),
    ]
    for c in neg_cols:
        missingness_exprs.append(F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c))
    quality_audit = channel_frame.agg(*missingness_exprs)


    # Explicit negative-delta policy diagnostics. The core default is null_invalid, but this compact table lets
    # reviewers see how floor-zero and keep-negative variants would change measured view mass by window.
    neg_policy_frames = []
    for window_days in ATTENTION_WINDOWS:
        prior_col = f"prior_lifetime_views_{window_days}d"
        elapsed_col = f"elapsed_days_{window_days}d"
        if prior_col not in channel_frame.columns or elapsed_col not in channel_frame.columns:
            continue
        delta = F.col("lifetime_views") - F.col(prior_col)
        elapsed = F.col(elapsed_col)
        valid_delta = (elapsed > 0) & delta.isNotNull()
        variants = {
            "null_invalid": F.when(valid_delta & (delta >= 0), delta / elapsed * F.lit(7.0)),
            "floor_zero": F.when(valid_delta, F.greatest(delta, F.lit(0.0)) / elapsed * F.lit(7.0)),
            "keep": F.when(valid_delta, delta / elapsed * F.lit(7.0)),
        }
        for policy_name, value_expr in variants.items():
            frame = channel_frame.withColumn("policy_weekly_views", value_expr).agg(
                F.count("*").alias("n_channels"),
                F.sum(F.when(F.col(prior_col).isNull(), 1).otherwise(0)).alias("n_missing_prior"),
                F.sum(F.when((elapsed > 0) & delta.isNull(), 1).otherwise(0)).alias("n_missing_delta_inputs"),
                F.sum(F.when(valid_delta & (delta < 0), 1).otherwise(0)).alias("n_negative_delta"),
                F.sum(F.when(F.col("policy_weekly_views").isNotNull(), 1).otherwise(0)).alias("n_measured_policy_weekly_views"),
                F.sum("policy_weekly_views").alias("sum_policy_weekly_views"),
            ).withColumn("window_days", F.lit(window_days)).withColumn("negative_delta_policy", F.lit(policy_name)).withColumn(
                "is_core_policy", F.lit(policy_name == NEGATIVE_DELTA_POLICY)
            )
            neg_policy_frames.append(frame)
    if neg_policy_frames:
        negative_delta_policy_summary = neg_policy_frames[0]
        for part in neg_policy_frames[1:]:
            negative_delta_policy_summary = negative_delta_policy_summary.unionByName(part, allowMissingColumns=True)
        write_delta(negative_delta_policy_summary, "robustness_negative_delta_policy")
        export_aggregate_csv(negative_delta_policy_summary, "robustness_negative_delta_policy")

    write_delta(quality_audit, "robustness_quality_audit")
    export_aggregate_csv(quality_audit, "robustness_quality_audit")
    display_if_enabled(quality_audit)
else:
    print('Skipped cell 37: execution_mode=manifest_only.')


## 12. Figure Rendering Helpers

These helpers read the aggregate Delta outputs, render figures, and save each figure as `.png`, `.pdf`, and `.svg` when supported. The code avoids raw row-level exports.


In [ ]:
NATURE_SINGLE_WIDTH = 90 / 25.4
NATURE_DOUBLE_WIDTH = 180 / 25.4
NATURE_TEXT_HEIGHT = 220 / 25.4

OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000"]
MUTED = ["#4C78A8", "#F58518", "#54A24B", "#E45756", "#72B7B2", "#B279A2", "#FF9DA6", "#9D755D", "#BAB0AC"]

mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


def panel_label(ax, label: str, x: float = -0.08, y: float = 1.04) -> None:
    ax.text(x, y, label, transform=ax.transAxes, fontweight="bold", fontsize=8, va="bottom", ha="left")


def save_figure(fig, name: str, tight: bool = True) -> None:
    safe = _safe_token(name)
    if tight:
        fig.tight_layout()
    for ext in ["png", "pdf", "svg"]:
        path = f"{FIG_DIR}/{safe}.{ext}"
        fig.savefig(path, bbox_inches="tight")
        record_manifest_output("figure", f"{safe}.{ext}", path, sha256=_sha256_of_file(path))
        print(f"Wrote {path}")
    plt.close(fig)


def aggregate_pdf(short_name: str, cols: Optional[Sequence[str]] = None, max_rows: int = 200000) -> pd.DataFrame:
    df = load_output(short_name)
    if cols:
        df = df.select(*cols)
    n = df.limit(max_rows + 1).count()
    if n > max_rows:
        raise ValueError(f"Aggregate {short_name} has {n:,}+ rows; raise max_rows only if this remains aggregate-safe.")
    return df.toPandas()


def unpack_quantiles(pdf: pd.DataFrame, col: str, prefix: str, probs: Sequence[float]) -> pd.DataFrame:
    arr = pdf[col].apply(lambda x: list(x) if isinstance(x, (list, tuple, np.ndarray)) else [np.nan] * len(probs))
    for idx, prob in enumerate(probs):
        pdf[f"{prefix}_p{int(prob * 100):02d}"] = arr.apply(lambda x: x[idx] if len(x) > idx else np.nan)
    return pdf


## 13. Render Main Figures


In [ ]:
if RUN_COMPUTE:
    # Figure 1: Discovery saturation / yield.
    disc = aggregate_pdf("discovery_by_batch")
    if not disc.empty and "batch_number" in disc.columns:
        fig, axes = plt.subplots(1, 2, figsize=(NATURE_DOUBLE_WIDTH, 2.6), sharex=False)
        summary = disc.groupby(["discovery_source", "batch_number"], dropna=False, as_index=False).agg({"new_channels": "sum", "new_lifetime_views": "sum"})
        for i, source in enumerate(summary["discovery_source"].dropna().unique()[:8]):
            part = summary[summary["discovery_source"] == source].sort_values("batch_number")
            axes[0].plot(part["batch_number"], part["new_channels"], marker="o", linewidth=1, markersize=2.5, label=source, color=MUTED[i % len(MUTED)])
        axes[0].set_yscale("symlog")
        axes[0].set_xlabel("Batch")
        axes[0].set_ylabel("New channels")
        axes[0].set_title("a  Discovery yield by batch")
        axes[0].legend(frameon=False, ncol=1)

        high = disc[disc["subscriber_band"].isin(["1M-10M", "10M+", "100k-1M"])]
        high_summary = high.groupby(["discovery_source", "batch_number"], dropna=False, as_index=False)["new_channels"].sum()
        for i, source in enumerate(high_summary["discovery_source"].dropna().unique()[:8]):
            part = high_summary[high_summary["discovery_source"] == source].sort_values("batch_number")
            axes[1].plot(part["batch_number"], part["new_channels"], marker="o", linewidth=1, markersize=2.5, label=source, color=MUTED[i % len(MUTED)])
        axes[1].set_yscale("symlog")
        axes[1].set_xlabel("Batch")
        axes[1].set_ylabel("New channels with >=100k subscribers")
        axes[1].set_title("b  High-attention discovery yield")
        save_figure(fig, "figure_1_discovery_saturation")
    else:
        print("Figure 1 skipped: discovery aggregate is empty.")
else:
    print('Skipped cell 41: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    # Figure 2: Treemap and composition panels.
    treemap_pdf = aggregate_pdf("treemap_cells", max_rows=50000)
    composition_pdf = aggregate_pdf("composition_by_traffic_block", max_rows=100000)

    if not treemap_pdf.empty:
        def _normalize_treemap_sizes(values: Sequence[float], dx: float, dy: float) -> List[float]:
            total = float(np.sum(values))
            return [float(v) * dx * dy / total for v in values] if total > 0 else []

        def _worst_ratio(row: Sequence[float], side: float) -> float:
            if not row or min(row) <= 0:
                return float("inf")
            s = sum(row)
            return max((side ** 2) * max(row) / (s ** 2), (s ** 2) / ((side ** 2) * min(row)))

        def _layout_row(row: Sequence[float], x: float, y: float, dx: float, dy: float) -> Tuple[List[dict], float, float, float, float]:
            rects = []
            row_sum = sum(row)
            if dx >= dy:
                height = row_sum / dx if dx else 0
                cx = x
                for size in row:
                    width = size / height if height else 0
                    rects.append({"x": cx, "y": y, "dx": width, "dy": height})
                    cx += width
                return rects, x, y + height, dx, dy - height
            width = row_sum / dy if dy else 0
            cy = y
            for size in row:
                height = size / width if width else 0
                rects.append({"x": x, "y": cy, "dx": width, "dy": height})
                cy += height
            return rects, x + width, y, dx - width, dy

        def _squarify(values: Sequence[float], x: float, y: float, dx: float, dy: float) -> List[dict]:
            sizes = [float(v) for v in _normalize_treemap_sizes(values, dx, dy) if float(v) > 0]
            rects, row = [], []
            while sizes:
                candidate = sizes[0]
                side = min(dx, dy)
                if not row or _worst_ratio(row + [candidate], side) <= _worst_ratio(row, side):
                    row.append(candidate)
                    sizes.pop(0)
                else:
                    new_rects, x, y, dx, dy = _layout_row(row, x, y, dx, dy)
                    rects.extend(new_rects)
                    row = []
            if row:
                new_rects, x, y, dx, dy = _layout_row(row, x, y, dx, dy)
                rects.extend(new_rects)
            return rects

        def _blend_with_white(hex_color: str, amount: float) -> tuple:
            hex_color = hex_color.lstrip("#")
            rgb = np.array([int(hex_color[i:i+2], 16) / 255 for i in (0, 2, 4)])
            return tuple(rgb * (1 - amount) + np.ones(3) * amount)

        tm = treemap_pdf.copy()
        tm["analysis_language"] = tm["analysis_language"].fillna("unknown").astype(str)
        tm["analysis_category"] = tm["analysis_category"].fillna("unknown").astype(str)
        tm["treemap_label"] = tm["treemap_label"].fillna("unknown").astype(str)
        tm["weekly_views"] = pd.to_numeric(tm["weekly_views"], errors="coerce").fillna(0.0)
        tm = tm[tm["weekly_views"] > 0].copy()
        tm["plot_label"] = np.where(tm["cell_type"].eq("channel"), tm["treemap_label"], "Other channels")

        language_totals = tm.groupby("analysis_language", as_index=False)["weekly_views"].sum().sort_values("weekly_views", ascending=False)
        language_order = language_totals["analysis_language"].tolist()
        language_color = {lang: MUTED[i % len(MUTED)] for i, lang in enumerate(language_order)}

        fig, ax = plt.subplots(figsize=(NATURE_DOUBLE_WIDTH, 5.0))
        panel_label(ax, "a", x=-0.015, y=1.015)
        lang_rects = _squarify(language_totals["weekly_views"].tolist(), 0.0, 0.0, 100.0, 60.0)
        total_tm = float(language_totals["weekly_views"].sum())

        for lang_rect, (_, lang_row) in zip(lang_rects, language_totals.iterrows()):
            lang = lang_row["analysis_language"]
            lx, ly, ldx, ldy = lang_rect["x"], lang_rect["y"], lang_rect["dx"], lang_rect["dy"]
            lang_frame = tm[tm["analysis_language"] == lang].copy()
            category_totals = lang_frame.groupby("analysis_category", as_index=False)["weekly_views"].sum().sort_values("weekly_views", ascending=False)
            cat_rects = _squarify(category_totals["weekly_views"].tolist(), lx, ly, ldx, ldy)
            base_color = language_color.get(lang, "#BAB0AC")
            for cat_idx, (cat_rect, (_, cat_row)) in enumerate(zip(cat_rects, category_totals.iterrows())):
                cat = cat_row["analysis_category"]
                cx, cy, cdx, cdy = cat_rect["x"], cat_rect["y"], cat_rect["dx"], cat_rect["dy"]
                cell_frame = lang_frame[lang_frame["analysis_category"] == cat].sort_values("weekly_views", ascending=False)
                cell_rects = _squarify(cell_frame["weekly_views"].tolist(), cx, cy, cdx, cdy)
                shade = min(0.55, 0.10 + 0.035 * (cat_idx % 10))
                for cell_rect, (_, cell_row) in zip(cell_rects, cell_frame.iterrows()):
                    chx, chy, chdx, chdy = cell_rect["x"], cell_rect["y"], cell_rect["dx"], cell_rect["dy"]
                    alpha_blend = shade if cell_row["cell_type"] == "channel" else min(0.75, shade + 0.22)
                    ax.add_patch(plt.Rectangle((chx, chy), chdx, chdy, facecolor=_blend_with_white(base_color, alpha_blend), edgecolor="white", linewidth=0.18))
                    cell_share = float(cell_row["weekly_views"]) / total_tm if total_tm else 0.0
                    if cell_row["cell_type"] == "channel" and cell_share >= 0.004 and chdx > 2.5 and chdy > 1.7:
                        ax.text(chx + 0.25, chy + 0.38, str(cell_row["plot_label"])[:28], fontsize=4.6, color="black", va="top", ha="left", clip_on=True)
                if cdx * cdy > 35:
                    ax.text(cx + 0.25, cy + cdy - 0.45, str(cat)[:24], fontsize=4.4, color="black", va="top", ha="left", alpha=0.85, clip_on=True)
            ax.add_patch(plt.Rectangle((lx, ly), ldx, ldy, fill=False, edgecolor="white", linewidth=0.9))
            if ldx * ldy > 85:
                ax.text(lx + ldx / 2, ly + ldy / 2, str(lang)[:18], ha="center", va="center", fontsize=min(8.5, 4.8 + ldx * ldy / 450), fontweight="bold", color="white", clip_on=True)

        ax.set_xlim(0, 100)
        ax.set_ylim(0, 60)
        ax.set_aspect("equal")
        ax.axis("off")
        ax.set_title("Public YouTube attention map: language to category to channel", pad=4)
        handles = [plt.Rectangle((0, 0), 1, 1, facecolor=language_color[lang], edgecolor="none") for lang in language_order[:10]]
        ax.legend(handles, language_order[:10], frameon=False, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.08), title="Largest language groups")
        save_figure(fig, "figure_2a_treemap_matplotlib", tight=False)

        try:
            import plotly.express as px
            fig_tm = px.treemap(
                tm,
                path=["analysis_language", "analysis_category", "treemap_label"],
                values="weekly_views",
                color="analysis_language",
                color_discrete_sequence=MUTED,
            )
            fig_tm.update_traces(textinfo="label+percent parent", marker_line_width=0.4)
            fig_tm.update_layout(margin=dict(l=2, r=2, t=24, b=2), font=dict(size=10, family="DejaVu Sans"))
            html_path = f"{FIG_DIR}/figure_2a_treemap_interactive.html"
            fig_tm.write_html(html_path, include_plotlyjs="cdn")
            print(f"Wrote {html_path}")
        except Exception as exc:
            print(f"Optional Plotly treemap HTML unavailable: {exc}")

    if not composition_pdf.empty:
        top_cats = composition_pdf.groupby("analysis_category", as_index=False)["weekly_views"].sum().sort_values("weekly_views", ascending=False).head(8)["analysis_category"].tolist()
        comp = composition_pdf.copy()
        comp["category_plot"] = np.where(comp["analysis_category"].isin(top_cats), comp["analysis_category"], "Other")
        comp = comp.groupby(["traffic_block", "category_plot"], as_index=False)["weekly_views"].sum()
        comp["share"] = comp["weekly_views"] / comp.groupby("traffic_block")["weekly_views"].transform("sum")
        wide = comp.pivot(index="traffic_block", columns="category_plot", values="share").fillna(0).sort_index()
        wide = wide[[c for c in top_cats if c in wide.columns] + (["Other"] if "Other" in wide.columns else [])]
        fig, ax = plt.subplots(figsize=(NATURE_DOUBLE_WIDTH, 2.8))
        bottom = np.zeros(len(wide))
        for i, col in enumerate(wide.columns):
            ax.bar(wide.index.astype(str), wide[col], bottom=bottom, label=col, color=MUTED[i % len(MUTED)], width=0.85)
            bottom += wide[col].to_numpy()
        ax.set_ylim(0, 1)
        ax.set_xlabel("Traffic block (1 = highest attention)")
        ax.set_ylabel("Share of weekly views")
        ax.set_title("Category composition across equal-attention traffic blocks")
        panel_label(ax, "b", x=-0.055, y=1.04)
        ax.legend(frameon=False, ncol=4, bbox_to_anchor=(0.5, -0.32), loc="upper center")
        save_figure(fig, "figure_2b_composition_blocks")
else:
    print('Skipped cell 42: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    # Figure 3: Subscriber proxy density, envelopes, and inactive/unmeasured share.
    density = aggregate_pdf("subscriber_proxy_density")
    envelope = aggregate_pdf("subscriber_proxy_envelope")
    inactive = aggregate_pdf("subscriber_proxy_inactive_share")
    if not density.empty and not envelope.empty:
        env = unpack_quantiles(envelope.copy(), "weekly_views_quantiles", "weekly_views", [0.05,0.10,0.25,0.50,0.75,0.90,0.95])
        env = unpack_quantiles(env, "views_per_sub_quantiles", "views_per_sub", [0.05,0.10,0.25,0.50,0.75,0.90,0.95])
        fig, axes = plt.subplots(1, 3, figsize=(NATURE_DOUBLE_WIDTH, 2.8), gridspec_kw={"width_ratios": [1.05, 1.05, 0.9]})
        pivot = density.pivot(index="log10_view_bin", columns="log10_sub_bin", values="n_channels").fillna(0)
        im = axes[0].imshow(np.log10(pivot.to_numpy() + 1), origin="lower", aspect="auto", cmap="viridis")
        axes[0].set_xlabel("log10 subscribers")
        axes[0].set_ylabel("log10 weekly views")
        axes[0].set_title("Density of active channels")
        xticks = np.linspace(0, max(len(pivot.columns) - 1, 0), min(6, len(pivot.columns))).astype(int) if len(pivot.columns) else np.array([], dtype=int)
        yticks = np.linspace(0, max(len(pivot.index) - 1, 0), min(6, len(pivot.index))).astype(int) if len(pivot.index) else np.array([], dtype=int)
        axes[0].set_xticks(xticks)
        axes[0].set_xticklabels([f"{float(pivot.columns[i]):.1f}" for i in xticks])
        axes[0].set_yticks(yticks)
        axes[0].set_yticklabels([f"{float(pivot.index[i]):.1f}" for i in yticks])
        fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="log10(n+1)")
        panel_label(axes[0], "a")

        axes[1].plot(env["log10_sub_bin"], env["weekly_views_p50"], color=OKABE_ITO[0], linewidth=1.4, label="median")
        axes[1].fill_between(env["log10_sub_bin"], env["weekly_views_p10"], env["weekly_views_p90"], color=OKABE_ITO[0], alpha=0.20, linewidth=0, label="10-90%")
        axes[1].fill_between(env["log10_sub_bin"], env["weekly_views_p25"], env["weekly_views_p75"], color=OKABE_ITO[0], alpha=0.35, linewidth=0, label="IQR")
        axes[1].set_yscale("symlog")
        axes[1].set_xlabel("log10 subscribers")
        axes[1].set_ylabel("Weekly views among active channels")
        axes[1].set_title("Conditional dispersion")
        axes[1].legend(frameon=False)
        panel_label(axes[1], "b")

        if not inactive.empty:
            inactive_plot = inactive.sort_values("log10_sub_bin")
            axes[2].plot(
                inactive_plot["log10_sub_bin"],
                inactive_plot["inactive_or_unmeasured_share"],
                color=OKABE_ITO[2],
                linewidth=1.4,
                marker="o",
                markersize=2.5,
            )
        axes[2].set_ylim(0, 1)
        axes[2].set_xlabel("log10 subscribers")
        axes[2].set_ylabel("Share inactive or unmeasured")
        axes[2].set_title("Inactive/unmeasured channels")
        panel_label(axes[2], "c")
        save_figure(fig, "figure_3_subscriber_proxy")
else:
    print('Skipped cell 43: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    # Figure 4: Threshold capture and Lorenz diagnostics.
    th = aggregate_pdf("threshold_capture")
    lor = aggregate_pdf("lorenz_curve")
    if not th.empty and not lor.empty:
        fig, axes = plt.subplots(1, 2, figsize=(NATURE_DOUBLE_WIDTH, 2.7))
        observed = th[th["status"].eq("observed_frame_share_not_platform_share")].copy() if "status" in th.columns else th.copy()
        if not observed.empty:
            observed["threshold_subscribers"] = pd.to_numeric(observed["threshold_subscribers"], errors="coerce")
            observed = observed[observed["threshold_subscribers"] > OBSERVED_ATTENTION_SUBSCRIBER_FLOOR].copy()
        below = th[th["status"].astype(str).str.contains("below_observed", na=False)].copy() if "status" in th.columns else pd.DataFrame()
        ax = axes[0]
        if not observed.empty:
            x = np.arange(len(observed))
            labels = [f"{int(v):,}" for v in observed["threshold_subscribers"]]
            ax.bar(x - 0.18, observed["share_channels"], width=0.36, color=OKABE_ITO[0], label="channels")
            ax.bar(x + 0.18, observed["share_weekly_views"], width=0.36, color=OKABE_ITO[1], label="weekly views")
            ax.set_xticks(x)
            ax.set_xticklabels(labels, rotation=30, ha="right")
            ax.legend(frameon=False)
        if not below.empty:
            ax.text(0.02, 0.04, "Sub-floor thresholds require the 1k strata / external estimator.", transform=ax.transAxes, fontsize=5.5, color="#555555", ha="left", va="bottom")
        ax.set_ylabel(f"Share of observed >= {OBSERVED_ATTENTION_SUBSCRIBER_FLOOR:,} frame")
        ax.set_xlabel("Subscriber threshold")
        ax.set_title("Threshold capture above observed floor")
        panel_label(ax, "a")

        axes[1].plot(lor["cum_channel_share"], lor["cum_view_share"], color=OKABE_ITO[0], linewidth=1.5)
        axes[1].plot([0, 1], [0, 1], color="#777777", linewidth=0.8, linestyle="--")
        axes[1].set_xlabel("Cumulative share of positive-attention channels")
        axes[1].set_ylabel("Cumulative share of weekly views")
        axes[1].set_title("Lorenz diagnostic")
        axes[1].set_xlim(0, 1)
        axes[1].set_ylim(0, 1)
        panel_label(axes[1], "b")
        save_figure(fig, "figure_4_threshold_capture")
else:
    print('Skipped cell 44: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    # Figure 5: Channel start year by traffic block.
    age = aggregate_pdf("age_histogram_by_traffic_block")
    age_summary_pdf = aggregate_pdf("age_summary_by_traffic_block")
    if not age.empty:
        plot_age = age[age["channel_start_year"].notna()].copy()
        plot_age["channel_start_year"] = plot_age["channel_start_year"].astype(int)
        plot_age = plot_age[(plot_age["channel_start_year"] >= 2005) & (plot_age["channel_start_year"] <= datetime.now().year)]
        top_blocks = sorted(plot_age["traffic_block"].dropna().unique())[:TRAFFIC_BLOCKS]
        pivot = plot_age.groupby(["traffic_block", "channel_start_year"], as_index=False)["weekly_views"].sum()
        pivot["share"] = pivot["weekly_views"] / pivot.groupby("traffic_block")["weekly_views"].transform("sum")
        wide = pivot.pivot(index="traffic_block", columns="channel_start_year", values="share").fillna(0).reindex(top_blocks)
        fig, ax = plt.subplots(figsize=(NATURE_DOUBLE_WIDTH, 3.0))
        im = ax.imshow(wide.to_numpy(), aspect="auto", cmap="mako" if "mako" in plt.colormaps() else "viridis")
        ax.set_yticks(np.arange(len(wide.index)))
        ax.set_yticklabels([str(int(v)) for v in wide.index])
        year_ticks = np.linspace(0, max(len(wide.columns)-1, 0), min(8, len(wide.columns))).astype(int)
        ax.set_xticks(year_ticks)
        ax.set_xticklabels([str(wide.columns[i]) for i in year_ticks], rotation=30, ha="right")
        ax.set_xlabel("Channel start year or earliest observed upload year")
        ax.set_ylabel("Traffic block")
        ax.set_title("Figure 5  Age structure across the attention distribution")
        fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="Share of block weekly views")
        save_figure(fig, "figure_5_age_structure")


    # Figure 5 alternative requested in manuscript notes: language rank, engagement, and speaker population.
    try:
        lang_pop = aggregate_pdf("language_rank_engagement_population", max_rows=5000)
    except Exception as exc:
        print(f"Language-rank population aggregate unavailable: {exc}")
        lang_pop = pd.DataFrame()

    if not lang_pop.empty:
        lp = lang_pop.sort_values("language_rank_by_attention").head(25).copy()
        lp["language_label"] = lp["analysis_language"].fillna("unknown").astype(str)
        fig, axes = plt.subplots(1, 2, figsize=(NATURE_DOUBLE_WIDTH, 3.2), gridspec_kw={"width_ratios": [1.15, 1.0]})
        ax = axes[0]
        colors = [MUTED[i % len(MUTED)] for i in range(len(lp))]
        ax.bar(lp["language_rank_by_attention"].astype(int).astype(str), lp["view_share"], color=colors, width=0.82)
        ax.set_xlabel("Language rank by weekly views")
        ax.set_ylabel("Share of weekly views")
        ax.set_ylim(0, max(0.01, float(lp["view_share"].max()) * 1.18))
        ax.tick_params(axis="x", rotation=90)
        panel_label(ax, "a")

        ax = axes[1]
        sc = lp.dropna(subset=["speaker_population_millions", "weekly_views_per_million_speakers"]).copy()
        if not sc.empty:
            sizes = np.clip(np.sqrt(sc["weekly_views"].to_numpy(dtype=float) / max(float(sc["weekly_views"].max()), 1.0)) * 260, 30, 260)
            ax.scatter(sc["speaker_population_millions"], sc["weekly_views_per_million_speakers"], s=sizes, color=OKABE_ITO[0], alpha=0.72, edgecolor="white", linewidth=0.4)
            for _, row in sc.head(12).iterrows():
                ax.text(row["speaker_population_millions"], row["weekly_views_per_million_speakers"], str(row["language_label"])[:14], fontsize=5.5, ha="left", va="bottom")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("Speaker population (millions, log)")
            ax.set_ylabel("Weekly views per million speakers (log)")
        else:
            ax.text(0.5, 0.5, "Speaker population table not configured", transform=ax.transAxes, ha="center", va="center")
            ax.set_axis_off()
        panel_label(ax, "b")
        fig.suptitle("Figure 5 alternative  Language rank and engagement", y=1.02, fontsize=8)
        save_figure(fig, "figure_5_language_rank_population")
else:
    print('Skipped cell 45: execution_mode=manifest_only.')


In [ ]:
if RUN_COMPUTE:
    # Figure 6: Production and format.
    prod = aggregate_pdf("production_summary_by_traffic_block_format")
    fmt = aggregate_pdf("format_share")
    if not prod.empty and not fmt.empty:
        fig, axes = plt.subplots(1, 2, figsize=(NATURE_DOUBLE_WIDTH, 2.8))
        p = prod.copy()
        p["uploads_per_channel"] = p["uploads"] / p["n_channels"].replace({0: np.nan})
        for i, fmt_name in enumerate(sorted(p["format"].dropna().unique())):
            part = p[p["format"] == fmt_name].sort_values("traffic_block")
            axes[0].plot(part["traffic_block"], part["uploads_per_channel"], marker="o", markersize=2.5, linewidth=1.2, label=fmt_name, color=OKABE_ITO[i % len(OKABE_ITO)])
        axes[0].set_xlabel("Traffic block")
        axes[0].set_ylabel(f"Uploads per channel in {PRODUCTION_WINDOW_DAYS}-day window")
        axes[0].set_title("a  Production intensity")
        axes[0].legend(frameon=False)

        share_col = "cumulative_recent_upload_view_share" if "cumulative_recent_upload_view_share" in fmt.columns else "video_view_share"
        axes[1].bar(fmt["format"].astype(str), fmt[share_col], color=OKABE_ITO[:len(fmt)])
        axes[1].set_ylim(0, 1)
        axes[1].set_ylabel(f"Share of cumulative views on uploads from last {FORMAT_LOOKBACK_DAYS} days")
        axes[1].set_xlabel("Format")
        axes[1].set_title("b  Recent-upload cumulative views")
        axes[1].tick_params(axis="x", rotation=20)
        save_figure(fig, "figure_6_production_format")
else:
    print('Skipped cell 46: execution_mode=manifest_only.')


## 14. Render Extended Data Figures

These figures are intentionally diagnostic. They can be moved, combined, or pruned as the main manuscript stabilizes.


In [ ]:
if RUN_COMPUTE:
    # ED Fig. 1: benchmark overlap and collection logs.
    try:
        bench = aggregate_pdf("benchmark_overlap")
        if not bench.empty:
            fig, ax = plt.subplots(figsize=(NATURE_SINGLE_WIDTH, 2.4))
            ax.barh(bench["benchmark"], bench["overlap_share"], color=OKABE_ITO[2])
            ax.set_xlim(0, 1)
            ax.set_xlabel("Overlap share with analysis frame")
            ax.set_title("ED Fig. 1  Benchmark overlap")
            save_figure(fig, "ed_figure_1_benchmark_overlap")
    except Exception as exc:
        print(f"ED Fig. 1 skipped: {exc}")

    # ED Fig. 2: category/language proxy heterogeneity.
    try:
        plc = aggregate_pdf("subscriber_proxy_by_language_category")
        if not plc.empty:
            top = plc.sort_values("weekly_views", ascending=False).head(30).copy()
            top["label"] = top["analysis_language"].astype(str) + " / " + top["analysis_category"].astype(str)
            fig, ax = plt.subplots(figsize=(NATURE_DOUBLE_WIDTH, 4.0))
            ax.barh(top["label"][::-1], top["pearson_log_subs_log_views"][::-1], color=OKABE_ITO[0])
            ax.set_xlim(-1, 1)
            ax.set_xlabel("log subscriber-view correlation")
            ax.set_title("ED Fig. 2  Subscriber proxy heterogeneity")
            save_figure(fig, "ed_figure_2_proxy_heterogeneity")
    except Exception as exc:
        print(f"ED Fig. 2 skipped: {exc}")

    # ED Fig. 3: concentration diagnostics.
    try:
        gini = aggregate_pdf("concentration_gini")
        rankcap = aggregate_pdf("rank_capture")
        if not rankcap.empty:
            order = ["top_100", "top_1k", "top_10k", "top_100k", "below_100k"]
            rankcap["rank_bucket"] = pd.Categorical(rankcap["rank_bucket"], categories=order, ordered=True)
            rankcap = rankcap.sort_values("rank_bucket")
            fig, ax = plt.subplots(figsize=(NATURE_SINGLE_WIDTH, 2.4))
            ax.bar(rankcap["rank_bucket"].astype(str), rankcap["view_share"], color=OKABE_ITO[1])
            ax.tick_params(axis="x", rotation=30)
            ax.set_ylabel("Weekly view share")
            title = "ED Fig. 3  Rank concentration"
            if not gini.empty:
                title += f" (Gini approx {gini.iloc[0,0]:.2f})"
            ax.set_title(title)
            save_figure(fig, "ed_figure_3_rank_concentration")
    except Exception as exc:
        print(f"ED Fig. 3 skipped: {exc}")

    # ED Fig. 4: major-language composition.
    try:
        ml = aggregate_pdf("major_language_traffic_blocks")
        if not ml.empty:
            major_langs = ml.groupby("analysis_language")["weekly_views"].sum().sort_values(ascending=False).head(6).index.tolist()
            fig, axes = plt.subplots(len(major_langs), 1, figsize=(NATURE_DOUBLE_WIDTH, max(2.0, 1.2 * len(major_langs))), sharex=True)
            if len(major_langs) == 1:
                axes = [axes]
            for ax, lang in zip(axes, major_langs):
                part = ml[ml["analysis_language"] == lang].copy()
                top_cats = part.groupby("analysis_category")["weekly_views"].sum().sort_values(ascending=False).head(5).index.tolist()
                part["category_plot"] = np.where(part["analysis_category"].isin(top_cats), part["analysis_category"], "Other")
                part = part.groupby(["traffic_block", "category_plot"], as_index=False)["weekly_views"].sum()
                part["share"] = part["weekly_views"] / part.groupby("traffic_block")["weekly_views"].transform("sum")
                wide = part.pivot(index="traffic_block", columns="category_plot", values="share").fillna(0).sort_index()
                bottom = np.zeros(len(wide))
                for i, col in enumerate(wide.columns):
                    ax.bar(wide.index.astype(str), wide[col], bottom=bottom, color=MUTED[i % len(MUTED)], width=0.85, label=col)
                    bottom += wide[col].to_numpy()
                ax.set_ylabel(str(lang))
            axes[0].set_title("ED Fig. 4  Composition by major language")
            axes[-1].set_xlabel("Traffic block")
            save_figure(fig, "ed_figure_4_language_composition")
    except Exception as exc:
        print(f"ED Fig. 4 skipped: {exc}")

    # ED Fig. 6/7 robustness panels.
    try:
        wrob = aggregate_pdf("robustness_attention_windows")
        if not wrob.empty:
            fig, ax = plt.subplots(figsize=(NATURE_SINGLE_WIDTH, 2.4))
            for i, threshold in enumerate(sorted(wrob["threshold_subscribers"].unique())):
                part = wrob[wrob["threshold_subscribers"] == threshold].sort_values("window_days")
                ax.plot(part["window_days"], part["view_share"], marker="o", linewidth=1.2, label=f">= {int(threshold):,}", color=OKABE_ITO[i % len(OKABE_ITO)])
            ax.set_xlabel("Attention window (days)")
            ax.set_ylabel("Threshold view share")
            ax.set_ylim(0, 1)
            ax.set_title("ED Fig. 6  Attention-window sensitivity")
            ax.legend(frameon=False)
            save_figure(fig, "ed_figure_6_attention_window_sensitivity")
    except Exception as exc:
        print(f"ED Fig. 6 skipped: {exc}")

    try:
        fs = aggregate_pdf("robustness_shorts_cutoff")
        if not fs.empty:
            fig, ax = plt.subplots(figsize=(NATURE_SINGLE_WIDTH, 2.4))
            labels = fs["shorts_cutoff_seconds"].astype(str) + "s / " + fs["format"].astype(str)
            ax.barh(labels, fs["view_share_within_cutoff"], color=OKABE_ITO[3])
            ax.set_xlim(0, 1)
            ax.set_xlabel("View share")
            ax.set_title("ED Fig. 7  Shorts cutoff sensitivity")
            save_figure(fig, "ed_figure_7_shorts_cutoff_sensitivity")
    except Exception as exc:
        print(f"ED Fig. 7 skipped: {exc}")
else:
    print('Skipped cell 48: execution_mode=manifest_only.')


## 15. Manuscript Tables and Source-Data Exports

This section writes compact source-data tables for manuscript figures. Keep these tables aggregate-only. Do not add channel-level or video-level exports here.


In [ ]:
if RUN_COMPUTE:
    source_data_manifest = []
    for short_name in [
        "traffic_block_summary",
        "attention_value_bin_map",
        "attention_observation_scope_summary",
        "attention_snapshot_date_summary",
        "attention_panel_depth_summary",
        "attention_anchor_snapshot_coverage",
        "attention_prior_snapshot_status",
        "attention_window_usage_summary",
        "attention_rank_status",
        "smoke_channel_sample_status",
        "anchor_date_candidates",
        "backfill_topic_category_coverage",
        "too_rank_attention_comparison",
        "too_rank_attention_comparison_by_band",
        "too_validation_layer_comparison",
        "discovery_by_batch",
        "benchmark_overlap",
        "composition_by_traffic_block",
        "language_category_composition",
        "language_market_jsd",
        "category_missingness_bounds",
        "major_language_traffic_blocks",
        "treemap_cells",
        "subscriber_proxy_summary",
        "subscriber_proxy_density",
        "subscriber_proxy_envelope",
        "subscriber_proxy_inactive_share",
        "threshold_capture",
        "rank_capture",
        "lorenz_curve",
        "lorenz_value_bins",
        "concentration_gini",
        "age_summary_by_traffic_block",
        "age_histogram_by_traffic_block",
        "language_rank_engagement_population",
        "production_summary_by_traffic_block_format",
        "format_share",
        "robustness_attention_windows",
        "robustness_suspicion_exclusion",
        "robustness_language_source_summary",
        "robustness_shorts_cutoff",
        "robustness_quality_audit",
        "robustness_negative_delta_policy",
        "sample_design_status",
        "residual_tail_bounds",
    ]:
        table = output_table(short_name)
        exists = table_exists(table)
        source_data_manifest.append({
            "analysis_run_id": RUN_ID,
            "short_name": short_name,
            "table_name": table,
            "exists": exists,
            "intended_use": "manuscript_source_data_or_diagnostic",
        })
    manifest_df = spark.createDataFrame(pd.DataFrame(source_data_manifest))
    write_delta(manifest_df, "source_data_manifest")
    export_aggregate_csv(manifest_df, "source_data_manifest")
    display_if_enabled(manifest_df, n=100)

    RUN_MANIFEST["finished_utc"] = datetime.now(timezone.utc).isoformat()
    RUN_MANIFEST["source_data_manifest_table"] = output_table("source_data_manifest")
    manifest_path = f"{OUTPUT_BASE_DIR}/{RUN_ID}/run_manifest_codex.json"
    with open(manifest_path, "w") as fh:
        json.dump(RUN_MANIFEST, fh, indent=2, default=str)
    record_manifest_output("run_manifest", "run_manifest_codex.json", manifest_path, sha256=_sha256_of_file(manifest_path))
    print(f"Wrote run manifest {manifest_path}")
else:
    print('Skipped cell 50: execution_mode=manifest_only.')


## 16. Interpretation Checklist for the Paper Draft

Use this checklist after a full run, before inserting numbers into the manuscript.

- **Discovery:** Does high-subscriber/high-view discovery yield approach zero across late batches? Does the benchmark overlap table support a plausible near-exhaustive head claim?
- **Composition:** Are unknown category/language shares small enough for a main-text whole-platform map? If not, move the weak axis to Extended Data until labels are improved.
- **Subscribers:** Report both central correlation and conditional dispersion. Avoid saying subscribers are useless; they are noisy proxies.
- **Thresholds:** Lead with one memorable, denominator-clear threshold fact. Keep Lorenz/Gini in Extended Data unless reviewers ask.
- **Age:** State exactly whether Figure 5 uses true channel creation date or earliest observed upload proxy.
- **Format:** Explicitly state the Shorts cutoff and the March 31, 2025 view-count rule issue in Methods.
- **Uncertainty:** Do not publish design-based lower-rank estimates until the sample-design widgets are filled and the design status table says `computed`.


## 17. Open Items Requiring Project Decisions

The notebook now encodes the current answers: preferred attention source is `dev_sean.default.yt_channel_stats_full`; category source is `dev_sean.default.backfill_channels.topic_categories`; residual-tail floor is 1,000 subscribers with the project estimator still external; suspicious-channel flags are diagnostic-only; and true channel creation dates remain unresolved. It no longer defaults to the likely partial `2026-05-28` partition. Blank `target_capture_date` now selects the latest partition passing the completeness rule and records the exact window used for every primary attention value.

1. What is the first complete Sunday-to-Sunday anchor pair after the large discovery run, and should manuscript runs require `attention_measure_status = primary_window_available` only?
2. `backfill_channels.topic_categories` appears sparse overall because many rows are still `pending`; category-missingness bounds are now written, but the publication treemap still needs the approved fallback hierarchy for the remaining channels.
3. What table/columns will hold the lower-rank stratified sample and weights once the 122M validation runs are complete?
4. Where will the 1k-subscriber tail estimator live, and what aggregate outputs should this notebook consume from it?
5. Do we have a true YouTube API `snippet.publishedAt` channel creation date anywhere in the warehouse, or should we add a backfill job? The current metadata inventory only exposes SocialBlade `general.created_at` for the top-list overlap.
6. Should the Fig. 5 main-panel choice be channel age or the JNS language-rank/population alternative now generated by this notebook?
7. Do we need a two-pass range-partitioned ranking implementation before 110M+ full-scale exact ranking, or is the observed head small enough for the current global exact-order window?
8. What is the approved public/export path for final aggregate source-data tables and figures inside the Databricks workspace?
